# Coin Flip Trading System

Uses a coin flip based trading strategy (heads = long, tails = short) to make trading decisions at market open while leveraging timeseries forcasting

## Module.1
### dependencies

In [21]:
# cellblock.1

# libraries
import numpy as np
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels
import yfinance as yf
import math
import re

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "font.size": 9})

In [ ]:
# cellblock.2

# import local data
#
# Only the PATH is needed here. The file is ~0.74 GB / 6.9M rows, and reading
# all of it just to print five lines costs a couple of GB that nothing
# downstream ever touches - Module.2 cellblock.1 re-reads it properly with
# usecols and narrow dtypes. So this cellblock peeks and stops.
import os

file_path = "/Users/andrew/main/trading/data/NQ.1m.OHLCV.data/NQ-2010.06.06-2026.01.23-ohlcv-1m.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"local /NQ data not found at {file_path}")

df = pd.read_csv(file_path, nrows=5)          # PREVIEW ONLY - not the dataset
print(f"{os.path.basename(file_path)}")
print(f"  size    : {os.path.getsize(file_path) / 1e9:.2f} GB on disk")
print(f"  columns : {list(df.columns)}")
print(df.head())


In [ ]:
# cellblock.3

# load yfinance data
YF_PERIOD   = "720d"        # just inside Yahoo's 730-day cap at 1h (~2 yrs)
YF_INTERVAL = "1h"
OHLCV = ["open", "high", "low", "close", "volume"]
WICK_RATIO = 1.2            # see _clip_dead_wicks


def _clip_dead_wicks(df, label, ratio=WICK_RATIO):
    """Clamp high/low to the bar's body on bars that report NO volume.

    Yahoo's pre/post bars carry volume 0, and their high/low are not
    trustworthy. On NVDA's 10-for-1 split day (2024-06-10) the 09:00 pre-market
    bar reads open 120.55, close 120.38, low 119.23 - and high 1208.88. That is
    the PRE-split price leaking into the wick. A short's stop sat 900 points
    away, the bogus number "hit" it, and the backtest booked a -$900.01 loss on
    a share that never traded above $123 that day. One bar, and NVDA's entire
    net went from about flat to -$1,008.

    The threshold is measured, not guessed. Across all eleven assets every wick
    beyond 1.2x the bar's own body sits on a zero-volume bar, and NOT ONE bar
    that actually traded exceeds it - futures and crypto top out at 1.13x. So
    this clamps defects only and can never touch a real range. It repairs rather
    than drops because the open and close of these bars are fine, and the open
    is what an entry uses; and it prints what it touched, because a silent
    repair is how bad data becomes a backtest result.
    """
    body_hi = df[["open", "close"]].max(axis=1)
    body_lo = df[["open", "close"]].min(axis=1).replace(0, np.nan)
    dead    = df["volume"] <= 0
    bad_hi  = dead & (df["high"] > body_hi * ratio)
    bad_lo  = dead & (df["low"] < body_lo / ratio)
    n = int((bad_hi | bad_lo).sum())
    if n:
        worst = float(pd.concat([df.loc[bad_hi, "high"] / body_hi[bad_hi],
                                 body_lo[bad_lo] / df.loc[bad_lo, "low"]]).max())
        df.loc[bad_hi, "high"] = body_hi[bad_hi]
        df.loc[bad_lo, "low"]  = body_lo[bad_lo]
        print(f"  [!] {label}: clamped {n} zero-volume bar(s) whose wick ran past "
              f"{ratio:g}x their own body (worst {worst:,.1f}x) - source defect")
    return df



def fetch_1h(ticker, label, period=YF_PERIOD):
    """Download max-available 1h bars and return a clean OHLCV frame.

    Returns all five OHLCV columns; index is tz-aware and sorted ascending.
    Yahoo serves at most 730 days at 1h against 60 days at 15m, so moving to the
    hourly interval buys ~12x the history - that depth is the reason to use it.
    720d is requested rather than 730d: it sits just inside the cap, so a
    boundary-rounding day at the far end cannot turn the whole request into an
    error. Yahoo often returns MORE than the window asked for anyway.
    """
    # prepost=True is what makes an equity tradeable by this system at all.
    # Yahoo's REGULAR-hours 1h bars for a stock start at 09:30, so nothing exists
    # in the 08:30-09:30 window the entry rule needs and every session is skipped.
    # With pre/post on, the 04:00-09:30 bars appear and the 09:00 bar lands in
    # the window. Verified to be a NO-OP for futures, crypto and ^VIX (identical
    # bar counts on NQ=F, ES=F, RTY=F, YM=F, BTC-USD, ETH-USD, ^VIX) - it only
    # changes the three equities, so it is safe to leave on for everything.
    # Caveat that matters: Yahoo reports pre/post VOLUME as 0. The prices are
    # real, the liquidity is not measurable here, and pre-market spreads are far
    # wider than the 1-tick slippage cellblock.2 assumes.
    raw = yf.download(ticker, period=period, interval=YF_INTERVAL,
                      auto_adjust=False, progress=False, threads=False,
                      prepost=True)
    if raw is None or len(raw) == 0:
        raise RuntimeError(f"{label} ({ticker}): yfinance returned no rows")

    # yfinance returns MultiIndex columns (field, ticker) -> flatten to field
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(-1)

    df = raw.rename(columns=str.lower)
    missing = [c for c in OHLCV if c not in df.columns]
    if missing:
        raise RuntimeError(f"{label} ({ticker}): missing columns {missing}")

    df = df[OHLCV].copy()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df.dropna(subset=["open", "high", "low", "close"])
    df["volume"] = df["volume"].fillna(0)
    df = _clip_dead_wicks(df, label)
    df.index.name = "ts"
    df.columns.name = None
    return df


def describe_1h(df, label, ticker):
    """Print coverage + integrity checks. Returns the frame unchanged."""
    span_days = (df.index[-1] - df.index[0]).total_seconds() / 86400
    bad_hl = int((df["high"] < df["low"]).sum())
    bad_rng = int((~((df["high"] >= df[["open", "close"]].max(axis=1) - 1e-6) &
                     (df["low"]  <= df[["open", "close"]].min(axis=1) + 1e-6))).sum())
    gap = df.index.to_series().diff().dt.total_seconds().div(60)
    print(f"{label}  ({ticker})  interval=1h")
    print(f"  bars      : {len(df):,}")
    print(f"  coverage  : {df.index[0]}  ->  {df.index[-1]}   ({span_days:.1f} calendar days)")
    print(f"  columns   : {list(df.columns)}")
    print(f"  close     : {df['close'].min():,.2f}  -  {df['close'].max():,.2f}")
    print(f"  volume    : total {df['volume'].sum():,.0f}"
          + ("   [!] all-zero: Yahoo reports no volume for this symbol"
             if df["volume"].sum() == 0 else ""))
    print(f"  integrity : high<low {bad_hl} | OHLC out of range {bad_rng} | NaN {int(df.isna().sum().sum())}")
    print(f"  bar gaps  : median {gap.median():.0f}m, max {gap.max():,.0f}m "
          f"(>60m gaps are session breaks/weekends)")
    return df


def plot_ohlcv(df, label, ticker, interval="1h", bar_minutes=60, downsampled=False):
    """Two-panel OHLCV chart: high-low range + close on top, volume beneath."""
    fig, (ax1, ax2) = plt.subplots(
        2, 1, sharex=True, figsize=(13, 6.5),
        gridspec_kw={"height_ratios": [3, 1]})

    ax1.fill_between(df.index, df["low"], df["high"], color="#4a7ebb",
                     alpha=0.30, linewidth=0, label="High-Low range")
    ax1.plot(df.index, df["close"], color="#14375e", linewidth=0.8, label="Close")
    ax1.plot(df.index, df["open"],  color="#c0504d", linewidth=0.5,
             alpha=0.55, label="Open")
    ax1.set_ylabel("Price")
    ax1.set_title(f"{label}  ({ticker})  -  {interval} OHLCV"
                  + ("  [chart downsampled]" if downsampled else "")
                  + f"  |  {len(df):,} bars  |  "
                  f"{df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}")
    ax1.legend(loc="upper left", fontsize=8, framealpha=0.9)

    width = bar_minutes / (24 * 60) * 0.9
    ax2.bar(df.index, df["volume"], width=width, color="#7f7f7f", linewidth=0)
    ax2.set_ylabel("Volume")
    ax2.set_xlabel("Time")
    if df["volume"].sum() == 0:
        ax2.text(0.5, 0.5, "no volume reported by source", ha="center",
                 va="center", transform=ax2.transAxes, fontsize=9, color="#c0504d")
    fig.autofmt_xdate()
    fig.subplots_adjust(hspace=0.08)
    plt.show()


def load_plot_1h(ticker, label):
    """Fetch -> validate -> chart. Returns the full 1h OHLCV frame."""
    df = fetch_1h(ticker, label)
    describe_1h(df, label, ticker)
    plot_ohlcv(df, label, ticker)
    return df


## Module.2
### load assets
**BTC, /ES, /NQ, /RTY, /YM, ETH, AAPL, NVDA, TSLA, VIX**

| cellblock | asset | source | interval | variable |
|---|---|---|---|---|
| 1 | /NQ | local CSV (front-month continuous) | **1m** | `nq_1m` |
| 2 | /NQ | yfinance `NQ=F` | 1h | `nq_1h` |
| 3 | /ES | yfinance `ES=F` | 1h | `es_1h` |
| 4 | /RTY | yfinance `RTY=F` | 1h | `rty_1h` |
| 5 | /YM | yfinance `YM=F` | 1h | `ym_1h` |
| 6 | BTC | yfinance `BTC-USD` | 1h | `btc_1h` |
| 7 | ETH | yfinance `ETH-USD` | 1h | `eth_1h` |
| 8 | TSLA | yfinance `TSLA` | 1h | `tsla_1h` |
| 9 | NVDA | yfinance `NVDA` | 1h | `nvda_1h` |
| 10 | AAPL | yfinance `AAPL` | 1h | `aapl_1h` |
| 11 | VIX | yfinance `^VIX` | 1h | `vix_1h` |

Every frame carries the full `open, high, low, close, volume` set. Yahoo caps 1h
intraday history at 730 days; `period="720d"` sits just inside that cap - about
12x the 60-day window it allows at 15m. Realised coverage varies by asset (crypto 24/7 > futures ~23h > equities RTH only).

In [ ]:
# cellblock.1

# /NQ local data (1m)
#
# Prerequisites, checked up front so a missing one says WHICH cell to run
# instead of surfacing as a bare NameError 40 lines in:
#   Module.1 cellblock.1 -> np, pd, plt, re
#   Module.1 cellblock.2 -> file_path
#   Module.1 cellblock.3 -> plot_ohlcv, OHLCV
_need = {"file_path": "Module.1 cellblock.2", "plot_ohlcv": "Module.1 cellblock.3",
         "OHLCV": "Module.1 cellblock.3", "re": "Module.1 cellblock.1",
         "np": "Module.1 cellblock.1", "pd": "Module.1 cellblock.1"}
_missing = {n: c for n, c in _need.items() if n not in globals()}
if _missing:
    raise RuntimeError("run these first -> " + "; ".join(
        f"{c} (defines {n})" for n, c in _missing.items()))

OUTRIGHT = re.compile(r"^NQ[HMUZ]\d$")          # NQ + month code + year digit
NQ_COLS  = ["ts_event", "instrument_id", "open", "high", "low", "close", "volume", "symbol"]

nq_raw = pd.read_csv(
    file_path, usecols=NQ_COLS, engine="pyarrow",
    dtype={"instrument_id": "int64", "open": "float32", "high": "float32",
           "low": "float32", "close": "float32", "volume": "int32",
           "symbol": "category"})

n_all = len(nq_raw)
nq_raw = nq_raw[nq_raw["symbol"].astype(str).str.fullmatch(OUTRIGHT)].copy()
print(f"rows: {n_all:,} total -> {len(nq_raw):,} outright "
      f"({n_all - len(nq_raw):,} spread rows dropped, {100*(n_all-len(nq_raw))/n_all:.2f}%)")

nq_raw["ts_event"] = pd.to_datetime(nq_raw["ts_event"], utc=True, format="ISO8601")
nq_raw["date"] = nq_raw["ts_event"].dt.date

# expiry order: each instrument_id is one real contract, so its last bar orders it
expiry_rank = {cid: k for k, cid in enumerate(
    nq_raw.groupby("instrument_id", observed=True)["ts_event"].max().sort_values().index)}
rank_to_id = {k: cid for cid, k in expiry_rank.items()}
print(f"distinct contracts (by instrument_id): {len(expiry_rank)}")

# front month = highest daily volume, constrained to never roll backwards
daily_vol = (nq_raw.groupby(["date", "instrument_id"], observed=True)["volume"]
                   .sum().reset_index())
front = (daily_vol.loc[daily_vol.groupby("date", observed=True)["volume"].idxmax(),
                       ["date", "instrument_id"]]
                  .rename(columns={"instrument_id": "front"})
                  .sort_values("date").reset_index(drop=True))
front["rank"] = front["front"].map(expiry_rank).astype(int)
front["rank_fwd"] = front["rank"].cummax()
print(f"days where raw volume leader moved backwards: {(front['rank'] != front['rank_fwd']).sum()}")
front["front_id"] = front["rank_fwd"].map(rank_to_id)
roll_dates = front.loc[front["front_id"] != front["front_id"].shift(), "date"].tolist()
print(f"roll events: {len(roll_dates)}  (expect ~4/yr over "
      f"{(nq_raw['ts_event'].max() - nq_raw['ts_event'].min()).days/365.25:.1f} yrs)")

# keep only bars belonging to that day's front contract -> continuous 1m OHLCV
nq_1m = (nq_raw.merge(front[["date", "front_id"]], on="date", how="left")
               .query("instrument_id == front_id")
               .sort_values("ts_event")
               .set_index("ts_event")[["open", "high", "low", "close", "volume", "symbol"]])
nq_1m.index.name = "ts"

# ---- integrity checks -------------------------------------------------------
logret = np.log(nq_1m["close"].astype("float64")).diff()
print(f"\n/NQ continuous front-month, 1m OHLCV")
print(f"  bars      : {len(nq_1m):,}")
print(f"  coverage  : {nq_1m.index[0]}  ->  {nq_1m.index[-1]}")
print(f"  columns   : {list(nq_1m.columns)}")
print(f"  close     : {nq_1m['close'].min():,.2f}  -  {nq_1m['close'].max():,.2f}")
print(f"  duplicate timestamps : {int(nq_1m.index.duplicated().sum())}")
ohlc_bad = int((~((nq_1m["high"] >= nq_1m[["open", "close"]].max(axis=1) - 1e-6) &
                  (nq_1m["low"] <= nq_1m[["open", "close"]].min(axis=1) + 1e-6))).sum())
print(f"  OHLC out of range    : {ohlc_bad}")
print(f"  max |1m log return|  : {logret.abs().max():.4f}"
      "   (roll gaps are NOT price-adjusted -- see note below)")
print(nq_1m.head())

# ---- chart ------------------------------------------------------------------
# 5.3M 1-minute points cannot be rendered legibly, so the CHART aggregates to
# daily bars. `nq_1m` itself keeps every 1-minute bar.
nq_daily = nq_1m.resample("1D").agg(
    {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
).dropna(subset=["close"])
plot_ohlcv(nq_daily, "/NQ  local front-month (1m source)", "NQ-2010.06.06-2026.01.23",
           interval="1m data, charted daily", bar_minutes=60*24, downsampled=True)

In [ ]:
# cellblock.2

# /NQ yfinance data (1h) - full OHLCV, maximum available history
nq_1h = load_plot_1h("NQ=F", "/NQ")
nq_1h.head()

In [ ]:
# cellblock.3

# /ES yfinance data (1h) - full OHLCV, maximum available history
es_1h = load_plot_1h("ES=F", "/ES")
es_1h.head()

In [ ]:
# cellblock.4

# /RTY yfinance data (1h) - full OHLCV, maximum available history
rty_1h = load_plot_1h("RTY=F", "/RTY")
rty_1h.head()

In [ ]:
# cellblock.5

# /YM yfinance data (1h) - full OHLCV, maximum available history
ym_1h = load_plot_1h("YM=F", "/YM")
ym_1h.head()

In [ ]:
# cellblock.6

# BTC yfinance data (1h) - full OHLCV, maximum available history
btc_1h = load_plot_1h("BTC-USD", "BTC")
btc_1h.head()

In [ ]:
# cellblock.7

# ETH yfinance data (1h) - full OHLCV, maximum available history
eth_1h = load_plot_1h("ETH-USD", "ETH")
eth_1h.head()

In [ ]:
# cellblock.8

# TSLA yfinance data (1h) - full OHLCV, maximum available history
tsla_1h = load_plot_1h("TSLA", "TSLA")
tsla_1h.head()

In [ ]:
# cellblock.9

# NVDA yfinance data (1h) - full OHLCV, maximum available history
nvda_1h = load_plot_1h("NVDA", "NVDA")
nvda_1h.head()

In [ ]:
# cellblock.10

# AAPL yfinance data (1h) - full OHLCV, maximum available history
aapl_1h = load_plot_1h("AAPL", "AAPL")
aapl_1h.head()

In [ ]:
# cellblock.11

# VIX yfinance data (1h) - full OHLCV, maximum available history
vix_1h = load_plot_1h("^VIX", "VIX")
vix_1h.head()

## Module.3
### Re-sampling

Builds every timeframe the strategy needs from the **finest real source available**, plus
maximum-history daily bars for long-horizon work.

| cellblock | builds |
|---|---|
| 1 | resampling engine - `resample_ohlcv()`, `describe_bars()` |
| 2 | fetchers - `fetch_daily()`, `fetch_1m()`, asset registry |
| 3 | `{a}_1d` - daily, maximum history, all 10 assets |
| 4 | `{a}_1m` - 1m bases (/NQ local CSV; others yfinance, chunked) |
| 5 | `{a}_3m` | 
| 6 | `{a}_5m` |
| 7 | `{a}_15m` |
| 8 | `{a}_30m` |
| 9 | `{a}_60m` |
| 10 | `DATASETS` registry + integrity / conservation checks |

Assets: **/NQ, /ES, /RTY, /YM, BTC, ETH, TSLA, NVDA, AAPL, VIX**

---

> **Why the intraday sets are NOT built from the daily bars.**
> Resampling only runs coarse-ward: `1m -> 15m` aggregates trades that actually printed.
> Going `1d -> 1m` is *upsampling* - it would have to invent ~1,380 intrabar prices per day
> that never existed. A backtest fed synthetic intrabar data fills orders at prices nobody
> could have transacted at, which silently manufactures edge. So daily is fetched and kept
> as its own long-history set, and the 1m..60m sets are aggregated **up** from 1-minute bars.
> `resample_ohlcv()` raises on any attempt to upsample.

> **Yahoo's hard limits** (probed, not assumed): `1m` = 8 days per request and ~30 days
> total; `5m/15m/30m` = 60 days; `1h` = 730 days; `1d` = full history. **`3m` is not a
> valid Yahoo interval at all** - it can only ever come from resampling 1m.

> **Known source defect:** Yahoo's *daily* futures bars include some where the open or
> close falls outside that bar's high/low range - /YM 29 bars (0.47%, worst 450 pts, as
> recent as 2025-03-19), /NQ 24, /ES 10; /RTY and the equities are clean. `fetch_daily`
> prints a warning but does **not** repair them, because silently rewriting prices is
> worse than knowing they are wrong. Filter them before using daily bars for fills.

> **`{a}_15m` is rebuilt here** from the 1m source and supersedes the native 15m pull from
> Module.2. For /NQ this is a large upgrade: 366k bars back to 2010 instead of Yahoo's
> 60-day window.

> **Bar convention:** bars are left-labelled (stamped at the bar's *open*), matching both the
> databento CSV and Yahoo. A bar stamped `09:30` on the 15m set covers `[09:30, 09:45)` and
> is therefore only complete at `09:45` - **do not let a signal read it at 09:30.**


In [ ]:
# cellblock.1

# coin flip strategy  -  signal generation only
#     heads -> long  (+1)
#     tails -> short (-1)
# This cellblock decides WHAT the position should be and WHEN that decision is
# allowed to become one. Sizing, costs, fills and P&L are NOT here.
#
# A fair coin has ZERO expected edge before costs and a strictly negative one
# after them. That is the point: this is the null hypothesis the rest of the
# system has to beat. If a backtest ever shows the raw flip making money, the
# backtest is broken - not the coin.
import hashlib

FLIP_SEED    = 20260918          # master seed - change it for a whole new history
HEADS, TAILS = "H", "T"

# Bar-step table, defined here so this cellblock runs standalone - it must not
# depend on any other cellblock in this module having executed first.
TIMEFRAMES = {"1m": "1min", "3m": "3min", "5m": "5min",
              "15m": "15min", "30m": "30min", "60m": "60min"}
BAR_STEPS  = {**TIMEFRAMES, "1d": "1D"}


def _bar_step(df, tf=None):
    """How long one bar lasts. A left-labelled bar is only complete this much later."""
    tf = tf or df.attrs.get("tf")
    if tf is not None:
        return pd.Timedelta(BAR_STEPS.get(tf, tf))
    if len(df) < 3:
        raise ValueError("cannot infer the bar step from < 3 bars - pass tf=")
    return pd.Timedelta(df.index.to_series().diff().median())


def _splitmix64(x):
    """64-bit avalanche. Same answer on every machine, every run, forever."""
    u = np.uint64
    with np.errstate(over="ignore"):
        x = x + u(0x9E3779B97F4A7C15)
        x = (x ^ (x >> u(30))) * u(0xBF58476D1CE4E5B9)
        x = (x ^ (x >> u(27))) * u(0x94D049BB133111EB)
        return x ^ (x >> u(31))


def _stream_key(label, seed):
    """Per-asset key. blake2b, not hash() - hash() is salted per interpreter run."""
    digest = hashlib.blake2b(str(label).encode(), digest_size=8,
                             person=b"coinflip").digest()
    with np.errstate(over="ignore"):
        return np.uint64(int.from_bytes(digest, "big")) ^ np.uint64(seed % 2**64)


def coin_flip_signals(df, label="", tf=None, seed=FLIP_SEED, p_heads=0.5,
                      stream="hash"):
    """Flip one coin per bar:  heads -> long (+1),  tails -> short (-1).

    Returns a frame on the SAME index as `df`:
        flip        H / T
        signal      +1 / -1  - the decision this bar produces
        valid_from  the first instant that decision may be acted on
        position    +1 / -1  - what is actually HELD during this bar

    `signal` and `position` are two columns on purpose. Bars are left-labelled,
    so the bar stamped 09:30 is not finished until 09:45; acting on its own
    signal at 09:30 trades on a bar that has not happened yet. `position` is the
    shifted, tradable version - downstream P&L must use `position`, never
    `signal`. The first `position` is NaN because no decision exists yet.

    stream="hash" (default): a bar's flip is a pure function of (seed, label, bar
    timestamp). Truncating, extending or reordering the sample cannot change any
    bar's flip, and a live session recomputes the same flip for the same bar
    without replaying history - so backtest, paper and live agree by construction.
    stream="sequential" draws one RNG stream in index order: reproducible, but
    every flip shifts if the start of the sample moves.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{label}: index must be a DatetimeIndex, "
                        f"got {type(df.index).__name__}")
    if not df.index.is_monotonic_increasing:
        raise ValueError(f"{label}: index must be sorted ascending")
    if df.index.duplicated().any():
        raise ValueError(f"{label}: {int(df.index.duplicated().sum())} duplicate "
                         "timestamps - one bar cannot get two flips")
    if not 0.0 < p_heads < 1.0:
        raise ValueError(f"{label}: p_heads must be in (0, 1), got {p_heads}")

    step = _bar_step(df, tf)

    if stream == "hash":
        with np.errstate(over="ignore"):
            bits = _splitmix64(df.index.asi8.astype(np.uint64)
                               ^ _stream_key(label, seed))
        draw = (bits >> np.uint64(11)).astype(np.float64) * 2.0**-53
    elif stream == "sequential":
        draw = np.random.default_rng([int(_stream_key(label, seed)),
                                      int(seed % 2**64)]).random(len(df))
    else:
        raise ValueError(f"unknown stream {stream!r} - use 'hash' or 'sequential'")

    heads = draw < p_heads

    out = pd.DataFrame(index=df.index.copy())
    out["flip"]       = pd.Categorical(np.where(heads, HEADS, TAILS),
                                       categories=[HEADS, TAILS])
    out["signal"]     = np.where(heads, 1, -1).astype("int8")
    out["valid_from"] = df.index + step
    out["position"]   = out["signal"].shift(1).astype("float64")
    out.index.name    = "ts"
    out.attrs.update(label=label, tf=tf or df.attrs.get("tf"), seed=seed,
                     p_heads=p_heads, stream=stream, bar_step=step)
    return out


def describe_signals(sig, label=None, tf=None):
    """Coverage + balance + turnover report for one signal set. Returns it unchanged."""
    label = label or sig.attrs.get("label", "")
    tf    = tf    or sig.attrs.get("tf") or ""
    n     = len(sig)
    nh    = int((sig["signal"] == 1).sum())
    p     = sig.attrs.get("p_heads", 0.5)
    # how many standard errors the realised heads rate sits from the coin's own p
    z     = (nh / n - p) / np.sqrt(p * (1 - p) / n) if n else float("nan")
    runs  = sig["signal"].ne(sig["signal"].shift()).cumsum().value_counts()
    held  = sig["position"].dropna()
    turns = int((held != held.shift()).iloc[1:].sum()) if len(held) > 1 else 0

    print(f"{label}  {tf}  coin flip (seed={sig.attrs.get('seed')}, "
          f"stream={sig.attrs.get('stream')})")
    print(f"  bars      : {n:,}")
    print(f"  coverage  : {sig.index[0]}  ->  {sig.index[-1]}")
    print(f"  flips     : {nh:,} heads / {n - nh:,} tails   "
          f"({100 * nh / n:.2f}% long, z={z:+.2f} vs p={p})")
    print(f"  runs      : {len(runs):,} same-way streaks, mean {runs.mean():.2f} bars, "
          f"longest {runs.max()}")
    print(f"  turnover  : {turns:,} position changes "
          f"({100 * turns / max(len(held) - 1, 1):.1f}% of bars, "
          f"~{2 * turns:,} contract-sides to trade before costs)")
    print(f"  tradable  : {int(sig['position'].notna().sum()):,} bars hold a position, "
          f"{int(sig['position'].isna().sum())} flat (no decision yet)")
    return sig


def check_signals(n=5_000, tf="15m", label="SELFTEST", seed=FLIP_SEED):
    """Falsifiable self-test on synthetic bars. One line per property, raises on failure.

    Test 3 is the one that matters: a coin flip reads no market data, so it
    CANNOT peek at the future. The test proves that claim rather than asserting
    it - rewrite every price and the flips must not move.
    """
    idx  = pd.date_range("2024-01-02 09:30", periods=n, freq=TIMEFRAMES[tf],
                         tz="America/New_York")
    rng  = np.random.default_rng(0)
    px   = 100 * np.exp(np.cumsum(rng.normal(0, 1e-3, n)))
    bars = pd.DataFrame({"open": px, "high": px * 1.001, "low": px * 0.999,
                         "close": px, "volume": rng.integers(1, 1e4, n)}, index=idx)
    bars.attrs["tf"] = tf

    sig    = coin_flip_signals(bars, label, tf=tf, seed=seed)
    checks = []

    # 1. position is the signal shifted one bar - the whole no-lookahead contract
    checks.append(("position == signal.shift(1), first bar flat",
                   bool((sig["position"].iloc[1:].to_numpy()
                         == sig["signal"].iloc[:-1].to_numpy()).all())
                   and bool(np.isnan(sig["position"].iloc[0]))))

    # 2. a decision is actionable no later than the bar it is applied in
    checks.append(("valid_from is after its own bar, at or before the next",
                   bool((sig["valid_from"] > sig.index).all())
                   and bool((sig["valid_from"].iloc[:-1].to_numpy()
                             <= sig.index[1:].to_numpy()).all())))

    # 3. flips do not depend on prices at all  ->  look-ahead is impossible
    scrambled = bars.copy()
    scrambled[["open", "high", "low", "close", "volume"]] *= rng.uniform(
        0.5, 2.0, size=(n, 5))
    checks.append(("flips unchanged when every price is rewritten",
                   bool((coin_flip_signals(scrambled, label, tf=tf, seed=seed)["signal"]
                         == sig["signal"]).all())))

    # 4. same seed -> same history, whatever the global RNG has been doing
    np.random.seed(1234)
    checks.append(("reproducible across calls",
                   bool((coin_flip_signals(bars, label, tf=tf, seed=seed)["signal"]
                         == sig["signal"]).all())))

    # 5. hash stream: a bar's flip does not move when the sample window moves
    k = n // 3
    checks.append(("window-invariant (hash stream)",
                   bool((coin_flip_signals(bars.iloc[k:], label, tf=tf, seed=seed)["signal"]
                         .to_numpy() == sig["signal"].iloc[k:].to_numpy()).all())))

    # 6. two assets must not share one coin, or they would trade in lockstep
    checks.append(("independent stream per label",
                   0.40 < float((coin_flip_signals(bars, "OTHER", tf=tf, seed=seed)["signal"]
                                 == sig["signal"]).mean()) < 0.60))

    # 7. a fair coin lands fair - |z| > 4 on 5k flips means a broken generator
    z = ((sig["signal"] == 1).mean() - 0.5) / np.sqrt(0.25 / n)
    checks.append((f"fair coin (z={z:+.2f}, |z| < 4)", abs(z) < 4))

    print(f"coin_flip_signals self-test  ({n:,} synthetic {tf} bars)")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise AssertionError(f"coin flip self-test FAILED: {failed}")
    print("  all properties hold - signals are reproducible and cannot see the future")
    return sig


_ = check_signals()

# Usage, once the bar sets exist:
#   nq_sig = coin_flip_signals(nq_15m, "/NQ", tf="15m")
#   describe_signals(nq_sig)
# Downstream P&L must read `position`, never `signal`.


In [ ]:
# cellblock.2

# entry / exit / risk  -  turns cellblock.1 flips into bounded round trips
#     size  : ALWAYS 1 unit - 1 E-mini contract, 1 share, or 1 coin
#     entry : one hour before the cash open, at that bar's open
#             heads -> long (+1),  tails -> short (-1)
#     stop  : placed exactly RISK_PCT of the account loss limit away
#             ($45,000 x 2% = $900), so one unit losing its stop costs $900
#     exit  : whichever comes first - the stop, or the cash close.
#             Nothing is ever carried overnight.
#
# Size is fixed, so the usual relationship inverts: position sizing does NOT
# absorb risk any more, the STOP DISTANCE does. One unit of a $20/point future
# hits $900 in 45 points; one share of a $250 stock needs a $900 move. The same
# rule therefore means something completely different per asset, and
# describe_trades() prints the stop as a % of price and as a fraction of ATR so
# that difference is impossible to miss.
#
# Read this before trusting any number that comes out of it:
#   - A stop is not a guarantee. It caps the loss only if the market trades
#     through it. Gap past it and the fill is the gap price, which is worse.
#     `planned_risk` is the intent; `worst` in the report is what happened.
#   - OHLC bars cannot say whether the low or the high came first inside a bar.
#     When a bar could have hit the stop, this code assumes it did. That is the
#     pessimistic reading and the only honest one at this resolution.
import math

# ---- account ---------------------------------------------------------------
# `loss_limit` is the number that ends the account - a prop firm's max loss, or
# the balance you refuse to go below. Every risk figure is a fraction OF THAT.
ACCOUNT = {
    "loss_limit":    45_000.0,   # the drawdown that blows the account
    "risk_pct":          0.02,   # <= 2% of it on any one trade  ->  $900
    "daily_loss_pct":    0.06,   # stop for the day after ~3 full stop-outs ($2,700)
}

UNIT_QTY = 1        # one mini contract / share / coin. Not a tunable.

# ---- instruments -----------------------------------------------------------
# point_value = dollars per 1.00 of price move, for ONE unit.
# fee = dollars per side per unit (futures). fee_bps = basis points of notional
# per side (crypto venues charge this way, and on a $90 stop it dominates).
# Verify every one of these against YOUR broker before believing any P&L.
INSTRUMENTS = {
    "/NQ":  dict(unit="1 E-mini contract", point_value=20.0, tick=0.25, fee=2.50, fee_bps=0.0,  tradable=True),
    "/ES":  dict(unit="1 E-mini contract", point_value=50.0, tick=0.25, fee=2.50, fee_bps=0.0,  tradable=True),
    "/RTY": dict(unit="1 E-mini contract", point_value=50.0, tick=0.10, fee=2.50, fee_bps=0.0,  tradable=True),
    "/YM":  dict(unit="1 E-mini contract", point_value=5.0,  tick=1.00, fee=2.50, fee_bps=0.0,  tradable=True),
    "BTC":  dict(unit="1 coin",            point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=10.0, tradable=True),
    "ETH":  dict(unit="1 coin",            point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=10.0, tradable=True),
    "TSLA": dict(unit="1 share",           point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=0.0,  tradable=True),
    "NVDA": dict(unit="1 share",           point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=0.0,  tradable=True),
    "AAPL": dict(unit="1 share",           point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=0.0,  tradable=True),
    "VIX":  dict(unit="index - NOT tradable", point_value=1.0, tick=0.01, fee=0.0, fee_bps=0.0, tradable=False),
    # You cannot hold the VIX index. VIXY is the ETF that tracks short-term VIX
    # futures and IS a share you can buy one of - so VIX stays here as the
    # untradable reference, and VIXY is what the strategy actually trades.
    "VIXY": dict(unit="1 share",           point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=0.0,  tradable=True),
}

# ---- session ---------------------------------------------------------------
# "one hour before market open" = cash_open - entry_lead, in exchange local time.
SESSION = {
    "tz":         "America/New_York",
    "cash_open":  "09:30",
    "cash_close": "16:00",
    "entry_lead": pd.Timedelta("1h"),
}


def risk_per_trade(account=None):
    """Dollars allowed to be lost on one trade. The 2% cap, in money."""
    a = {**ACCOUNT, **(account or {})}
    return a["loss_limit"] * a["risk_pct"]


def stop_distance(symbol, account=None):
    """How far the stop sits, in points, for ONE unit.

    Size is fixed at 1, so this is the only lever left: distance = budget /
    point_value. Rounded DOWN to a whole tick, because rounding up would widen
    the stop past the 2% cap that the whole rule exists to enforce.
    """
    spec = INSTRUMENTS[symbol]
    raw  = risk_per_trade(account) / spec["point_value"]
    return math.floor(raw / spec["tick"]) * spec["tick"]


def wilder_atr(bars, n=14):
    """Wilder's ATR in points. Value at bar t INCLUDES bar t - shift before use.

    Not used to place the stop - the stop comes from the risk budget. This is
    here so the report can say how the stop compares with normal bar movement.
    """
    pc = bars["close"].shift(1)
    tr = pd.concat([bars["high"] - bars["low"],
                    (bars["high"] - pc).abs(),
                    (bars["low"]  - pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()


def round_turn_fees(spec, entry_px, exit_px, qty=UNIT_QTY):
    """Commission both sides, plus notional-based venue fees where they apply."""
    flat = 2.0 * spec["fee"] * qty
    bps  = (abs(entry_px) + abs(exit_px)) * qty * spec["point_value"] \
           * spec["fee_bps"] / 10_000.0
    return flat + bps


class RiskBook:
    """Account-level guard rails. The kill switch lives here.

    Three layers, checked before every entry:
      per trade : the stop is placed at exactly the 2% budget
      per day   : daily_loss_pct of the loss limit, then no more trades today
      account   : drawdown from peak equity >= loss_limit  ->  KILLED, forever
    """

    def __init__(self, account=None):
        a = {**ACCOUNT, **(account or {})}
        self.loss_limit = a["loss_limit"]
        self.daily_cap  = a["loss_limit"] * a["daily_loss_pct"]
        self.equity = self.peak = self.day_pnl = 0.0
        self.day = None
        self.killed, self.kill_ts = False, None

    def roll_to(self, day):
        """New session -> the daily loss budget resets."""
        if day != self.day:
            self.day, self.day_pnl = day, 0.0

    def blocked(self):
        """Why trading is not allowed right now, or None."""
        if self.killed:
            return "kill switch (account loss limit)"
        if self.day_pnl <= -self.daily_cap:
            return "daily loss limit"
        return None

    def record(self, pnl, ts):
        """Book a closed trade and re-check the kill switch."""
        self.equity  += pnl
        self.day_pnl += pnl
        self.peak = max(self.peak, self.equity)
        if not self.killed and (self.peak - self.equity) >= self.loss_limit:
            self.killed, self.kill_ts = True, ts


def run_session_trades(bars, sig, symbol, account=None, session=None,
                       slip_ticks=1.0, allow_untradable=False):
    """One unit, one trade per session: in an hour before the open, out by the close.

    Returns (trades, skips, book).

    No-lookahead, specifically:
      side      comes from sig["position"], which cellblock.1 already shifted -
                it is the flip of a bar that had CLOSED before this entry.
      entry_px  is the entry bar's OPEN, the first price of that bar.
      stop      is a constant derived from the account, not from any bar, so
                nothing inside the entry bar can move it.
    """
    account = {**ACCOUNT, **(account or {})}
    session = {**SESSION, **(session or {})}
    if symbol not in INSTRUMENTS:
        raise KeyError(f"{symbol} not in INSTRUMENTS - add its point value first")
    spec = INSTRUMENTS[symbol]
    if not spec["tradable"] and not allow_untradable:
        raise ValueError(f"{symbol} is {spec['unit']} - you cannot hold a unit of it. "
                         "Trade a future or ETF on it instead, or pass "
                         "allow_untradable=True to model it anyway.")
    if not bars.index.equals(sig.index):
        raise ValueError("bars and signals must share one index - build the "
                         "signals from these exact bars")
    if bars.index.tz is None:
        # Never localize silently. The session rules are wall-clock, so guessing
        # the zone would move every entry by whole hours and the backtest would
        # still "work" - it would just be trading the wrong bar.
        raise ValueError(
            f"{symbol}: bar index is tz-naive, so it cannot be placed on a "
            f"{session['tz']} trading session. yfinance returns tz-naive stamps "
            "for DAILY bars and tz-aware ones for intraday, so a 1d set lands "
            "here naive. Fix it at the source, or localize deliberately:\n"
            '    bars.index = bars.index.tz_localize("America/New_York")\n'
            "and only if you know that is the zone the stamps are already in.")

    budget      = risk_per_trade(account)
    stop_points = stop_distance(symbol, account)
    slip        = slip_ticks * spec["tick"]
    pv          = spec["point_value"]
    tz          = session["tz"]

    et   = bars.index.tz_convert(tz)
    o, h = bars["open"].to_numpy(), bars["high"].to_numpy()
    l, c = bars["low"].to_numpy(),  bars["close"].to_numpy()
    pos  = sig["position"].to_numpy()
    atr  = wilder_atr(bars).shift(1).to_numpy()          # context only

    book, trades, skips = RiskBook(account), [], []

    # Session bounds by binary search on the sorted index, and the stop scan
    # vectorised inside the session. The naive form rescans every bar for every
    # day - fine at 1h, hopeless on a 5M-bar 1m series (hours vs seconds).
    for _midnight in et.normalize().unique():
        day      = _midnight.date()
        open_ts  = pd.Timestamp(f"{day} {session['cash_open']}",  tz=tz)
        close_ts = pd.Timestamp(f"{day} {session['cash_close']}", tz=tz)
        target   = open_ts - session["entry_lead"]
        book.roll_to(day)

        # the entry bar: first bar at or after the target, still before the open
        lo = int(et.searchsorted(target,  "left"))
        hi = int(et.searchsorted(open_ts, "left"))
        if hi <= lo:
            skips.append(dict(day=day, reason="no bar in the hour before the open"))
            continue
        i = lo

        why = book.blocked()
        if why:
            skips.append(dict(day=day, reason=why))
            continue
        if not np.isfinite(pos[i]) or pos[i] == 0:
            skips.append(dict(day=day, reason="no tradable flip yet"))
            continue

        side     = int(pos[i])
        entry_px = o[i] + side * slip                  # pay up to get in
        stop_px  = entry_px - side * stop_points

        # exit scan - the entry bar counts, we are in it from its open
        end = max(int(et.searchsorted(close_ts, "right")), i + 1)
        hit = (l[i:end] <= stop_px) if side > 0 else (h[i:end] >= stop_px)
        if hit.any():
            exit_i  = i + int(hit.argmax())
            # gapped through? then the fill is the gap, not the stop
            exit_px = min(o[exit_i], stop_px) if side > 0 else max(o[exit_i], stop_px)
            reason  = "stop"
        else:
            exit_i, exit_px, reason = end - 1, c[end - 1], "cash close"
        exit_px -= side * slip                         # give up edge to get out

        gross = side * (exit_px - entry_px) * pv * UNIT_QTY
        fees  = round_turn_fees(spec, entry_px, exit_px)
        net   = gross - fees
        book.record(net, bars.index[exit_i])

        trades.append(dict(
            day=day, entry_ts=bars.index[i], exit_ts=bars.index[exit_i],
            side=side, flip="H" if side > 0 else "T", qty=UNIT_QTY,
            entry_px=entry_px, stop_px=stop_px, exit_px=exit_px,
            stop_points=stop_points, planned_risk=stop_points * pv * UNIT_QTY,
            atr=atr[i], exit_reason=reason,
            gross=gross, fees=fees, net=net, equity=book.equity))

    tr = pd.DataFrame(trades)
    if len(tr):
        tr = tr.set_index("entry_ts")
    sk = pd.DataFrame(skips)
    tr.attrs.update(symbol=symbol, budget=budget, stop_points=stop_points,
                    account=account, session=session, slip_ticks=slip_ticks)
    return tr, sk, book


def describe_trades(trades, skips, book, symbol, account=None):
    """Did the risk rules hold, and what did the flip actually cost? Returns trades."""
    a      = {**ACCOUNT, **(account or {})}
    budget = risk_per_trade(a)
    spec   = INSTRUMENTS[symbol]
    pts    = stop_distance(symbol, a)
    n      = len(trades)

    print(f"{symbol}  {spec['unit']}  -  ${budget:,.0f} risk "
          f"({a['risk_pct']:.0%} of ${a['loss_limit']:,.0f})  ->  stop {pts:,.2f} pts")
    if n == 0:
        print("  trades    : NONE")
        if len(skips):
            for why, k in skips["reason"].value_counts().items():
                print(f"      {k:>5}  {why}")
        return trades

    wins  = trades["net"] > 0
    dd    = (trades["equity"].cummax() - trades["equity"]).max()
    worst = trades["net"].min()
    pct   = 100 * pts / trades["entry_px"].mean()
    xatr  = pts / trades["atr"].mean() if trades["atr"].notna().any() else float("nan")

    print(f"  trades    : {n:,}  ({int(wins.sum())} up / {int((~wins).sum())} down, "
          f"{100 * wins.mean():.1f}% win rate)")
    print(f"  stop      : {pts:,.2f} pts = {pct:.3f}% of price = {xatr:.2f}x ATR(14)"
          + ("   [!] inside one bar's normal range - noise will hit it" if xatr < 1
             else "   [!] wider than any plausible day - it will rarely trigger"
             if xatr > 10 else ""))
    print(f"  exits     : " + ", ".join(
        f"{k} {v}" for k, v in trades["exit_reason"].value_counts().items()))
    print(f"  gross     : ${trades['gross'].sum():>12,.2f}")
    print(f"  fees      : ${-trades['fees'].sum():>12,.2f}   "
          f"({trades['fees'].mean():,.2f}/trade = "
          f"{100 * trades['fees'].mean() / budget:.1f}% of the risk budget)")
    print(f"  NET       : ${trades['net'].sum():>12,.2f}   "
          f"({trades['net'].mean():+,.2f}/trade)")
    print(f"  max DD    : ${dd:,.2f} of the ${a['loss_limit']:,.0f} limit "
          f"({100 * dd / a['loss_limit']:.1f}%)")
    breach = trades["planned_risk"] > budget + 1e-9
    print(f"  risk cap  : planned risk <= ${budget:,.0f} on {n - int(breach.sum()):,}/{n:,}"
          + ("  [OK]" if not breach.any() else f"  [!] {int(breach.sum())} BREACH"))
    print(f"  worst     : ${worst:,.2f} realised vs ${budget:,.0f} planned"
          + ("   [!] a gap beat the stop" if worst < -(budget + trades['fees'].max())
             else "   (no gap beat the stop)"))
    print(f"  kill sw   : " + (f"TRIPPED {book.kill_ts}" if book.killed else "not tripped"))
    if len(skips):
        print(f"  skipped   : {len(skips):,} sessions")
        for why, k in skips["reason"].value_counts().head(4).items():
            print(f"      {k:>5}  {why}")
    return trades


def run_all(datasets, account=None, verbose=True, **kw):
    """Apply the same rules to every asset. One unit each. Returns (summary, book_of_trades).

    `datasets` is {label: bars}. Untradable labels (VIX) are reported, not run.
    """
    rows, out = [], {}
    for label, bars in datasets.items():
        spec = INSTRUMENTS.get(label)
        if spec is None:
            rows.append(dict(asset=label, unit="?", trades=0, note="not in INSTRUMENTS"))
            continue
        if not spec["tradable"]:
            rows.append(dict(asset=label, unit=spec["unit"], trades=0,
                             note="cannot hold a unit - skipped"))
            continue
        sig = coin_flip_signals(bars, label, tf=bars.attrs.get("tf"))
        tr, sk, book = run_session_trades(bars, sig, label, account=account, **kw)
        out[label] = (tr, sk, book)
        if verbose:
            describe_trades(tr, sk, book, label, account)
            print()
        if len(tr) == 0:
            note = sk["reason"].mode()[0] if len(sk) else "no sessions"
            rows.append(dict(asset=label, unit=spec["unit"], trades=0, note=note))
            continue
        rows.append(dict(
            asset=label, unit=spec["unit"], trades=len(tr),
            stop_pts=tr["stop_points"].iloc[0],
            stop_pct=100 * tr["stop_points"].iloc[0] / tr["entry_px"].mean(),
            x_atr=tr["stop_points"].iloc[0] / tr["atr"].mean(),
            win_pct=100 * (tr["net"] > 0).mean(),
            stop_pct_exits=100 * (tr["exit_reason"] == "stop").mean(),
            gross=tr["gross"].sum(), fees=-tr["fees"].sum(), net=tr["net"].sum(),
            max_dd=(tr["equity"].cummax() - tr["equity"]).max(),
            killed=book.killed, note=""))
    return pd.DataFrame(rows).set_index("asset"), out


def check_entries_exits(days=90, symbol="/NQ", seed=FLIP_SEED):
    """Falsifiable self-test. One line per rule, raises if any rule is broken."""
    tz  = SESSION["tz"]
    idx = pd.date_range("2024-01-01 00:00", periods=days * 24, freq="60min", tz=tz)
    idx = idx[idx.dayofweek < 5]
    rng = np.random.default_rng(7)
    px  = 18_000 * np.exp(np.cumsum(rng.normal(0, 1.5e-3, len(idx))))
    rad = np.abs(rng.normal(0, 12, len(idx))) + 3
    bars = pd.DataFrame({"open": px, "high": px + rad, "low": px - rad,
                         "close": px + rng.normal(0, 6, len(idx)),
                         "volume": rng.integers(1, 9999, len(idx))}, index=idx)
    bars["high"] = bars[["open", "high", "close"]].max(axis=1)
    bars["low"]  = bars[["open", "low",  "close"]].min(axis=1)
    bars.attrs["tf"] = "60m"

    sig = coin_flip_signals(bars, symbol, tf="60m", seed=seed)
    tr, sk, book = run_session_trades(bars, sig, symbol)
    if len(tr) == 0:
        raise AssertionError("self-test produced no trades - cannot verify anything")

    budget = risk_per_trade()
    et_in  = tr.index.tz_convert(tz)
    et_out = pd.DatetimeIndex(tr["exit_ts"]).tz_convert(tz)
    opens  = pd.to_datetime([f"{d} {SESSION['cash_open']}"  for d in tr["day"]]).tz_localize(tz)
    closes = pd.to_datetime([f"{d} {SESSION['cash_close']}" for d in tr["day"]]).tz_localize(tz)
    checks = []

    # 1. size is one unit. Never two, never zero, never fractional.
    checks.append(("size is exactly 1 unit on every trade",
                   bool((tr["qty"] == 1).all())))

    # 2. the 2% cap - with size fixed, the stop distance is what enforces it
    checks.append((f"planned risk <= ${budget:,.0f} on every trade",
                   bool((tr["planned_risk"] <= budget + 1e-9).all())))

    # 3. entry lands in [one hour before the open, the open)
    checks.append(("entry in the hour before the cash open",
                   bool(((et_in >= opens - SESSION["entry_lead"]) & (et_in < opens)).all())))

    # 4. nothing is carried overnight
    checks.append(("flat by the cash close, every session",
                   bool((et_out <= closes).all())
                   and bool((pd.Index(et_out.date) == pd.Index(tr["day"])).all())))

    # 5. heads -> long, tails -> short, using the SHIFTED decision
    want = sig["position"].reindex(tr.index).to_numpy()
    checks.append(("side == the flip already on the book (heads long, tails short)",
                   bool((tr["side"].to_numpy() == want).all())
                   and bool((np.where(tr["flip"] == "H", 1, -1) == tr["side"]).all())))

    # 6. nothing inside the entry bar may move the stop  ->  no look-ahead
    poke  = bars.copy()
    first = bars.index.get_loc(tr.index[0])
    poke.iloc[first, poke.columns.get_loc("high")] *= 1.05
    poke.iloc[first, poke.columns.get_loc("low")]  *= 0.95
    tr2, _, _ = run_session_trades(poke, coin_flip_signals(poke, symbol, tf="60m", seed=seed),
                                   symbol)
    checks.append(("stop unmoved when the entry bar's own high/low are rewritten",
                   bool(abs(tr2["stop_px"].iloc[0] - tr["stop_px"].iloc[0]) < 1e-9)))

    # 7. a stop fill is never BETTER than the stop - gaps fill worse, not free
    st = tr[tr["exit_reason"] == "stop"]
    checks.append((f"stop fills never better than the stop ({len(st)} stop-outs)",
                   bool((st["side"] * (st["exit_px"] - st["stop_px"]) <= 1e-9).all())))

    # 8. an untradable instrument must be refused, not silently modelled
    try:
        run_session_trades(bars, sig, "VIX")
        refused = False
    except ValueError:
        refused = True
    checks.append(("refuses to trade a unit of an index (VIX)", refused))

    # 9. once the kill switch trips, the account never trades again
    tiny = {"loss_limit": 300.0, "risk_pct": 0.30, "daily_loss_pct": 10.0}
    tr3, _, bk3 = run_session_trades(bars, sig, symbol, account=tiny)
    checks.append(("kill switch blocks every entry after it trips",
                   (not bk3.killed) or bool((pd.DatetimeIndex(tr3["exit_ts"])
                                             <= bk3.kill_ts).all())))

    print(f"entry / exit / risk self-test  ({len(tr):,} trades on {days} synthetic days)")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise AssertionError(f"risk self-test FAILED: {failed}")
    print("  every entry, exit and risk rule holds on synthetic data")
    return tr


_ = check_entries_exits()

# Usage, once cellblock.1 has made the flips:
#   sig          = coin_flip_signals(nq_1h, "/NQ", tf="60m")
#   tr, sk, book = run_session_trades(nq_1h, sig, "/NQ")
#   describe_trades(tr, sk, book, "/NQ")
#
#   summary, all_trades = run_all({"/NQ": nq_1h, "BTC": btc_1h, "AAPL": aapl_1h})
#   summary


In [ ]:
# cellblock.3

# P&L visuals  -  what the rules in cellblock.2 actually did to the account
#     collect_assets : every {asset}_{tf} frame this notebook has built
#     plot_pnl       : equity + drawdown vs the loss limit, one dataset
#     plot_pnl_fan   : one dataset under COIN_SEEDS coins - the dispersion chart
#     plot_pnl_grid  : small multiples, EVERY dataset, one equity panel each
#     plot_fan_grid  : small multiples, EVERY dataset, COIN_SEEDS coins each
#
# Two rules the grids follow, both about not lying by omission:
#   - A dataset that cannot trade still gets a panel, with the reason printed in
#     it. Dropping it would read as "no result"; "no bar in the hour before the
#     open" reads as "this data cannot answer the question", which is the truth.
#   - Panels do NOT share an x axis, because the datasets do not share a period:
#     /NQ 1m is 15.6 years, the Yahoo 1h sets are ~2. Each panel therefore
#     carries its own date span in the title - compare the shapes, not the
#     widths, and trust the long sample over the short ones.
#
# Every function RETURNS the frame it drew, so each figure has a table behind it.

COIN_SEEDS = 50     # how many coins every dispersion chart draws

# label -> the notebook's variable stem. Timeframes are discovered, not assumed,
# so whatever Module.2/3 has built shows up without editing this cell.
VAR_OF = {"/NQ": "nq", "/ES": "es", "/RTY": "rty", "/YM": "ym",
          "BTC": "btc", "ETH": "eth", "TSLA": "tsla", "NVDA": "nvda",
          "AAPL": "aapl", "VIX": "vix", "VIXY": "vixy"}
# 1d is deliberately absent: the entry rule needs a bar INSIDE the hour before
# the cash open, and a daily bar has no inside. Every 1d set would render as an
# empty panel, so they are excluded here rather than drawn as ten blanks.
TF_ORDER = ["1m", "3m", "5m", "15m", "30m", "60m", "1h"]
GRID_TFS      = ["1m", "1h"]   # what the grids use unless told otherwise
MIN_SPAN_DAYS = 180            # shorter than this is not a sample, it is an anecdote

# "$" is data in every label on these charts, not a LaTeX delimiter. Left on,
# matplotlib reads the text between two "$" as mathtext and silently eats the
# spacing ("$-45,690 to +169,830" renders as "-45,690to+169,830").
plt.rcParams["text.parse_math"] = False

# Chart tokens. Categorical slot 1 (blue) for the series, status-critical for
# loss. Validated as a pair against both surfaces: CVD dE 23.8 light / 25.7 dark,
# normal-vision 31.6 / 31.9, both >= 3:1 on their surface.
VIZ = {
    "light": dict(surface="#fcfcfb", ink="#0b0b0b", secondary="#52514e",
                  muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
                  series="#2a78d6", loss="#d03b3b", ensemble="#898781"),
    "dark":  dict(surface="#1a1a19", ink="#ffffff", secondary="#c3c2b7",
                  muted="#898781", grid="#2c2c2a", axis="#383835",
                  series="#3987e5", loss="#d03b3b", ensemble="#898781"),
}


def collect_assets(scope=None, tfs=GRID_TFS, min_days=MIN_SPAN_DAYS, quiet=False):
    """Every {asset}_{tf} frame in the notebook -> {panel label: (symbol, bars)}.

    Discovers what exists rather than demanding a fixed list, so /NQ's 15.6-year
    local 1m set and the ~2-year Yahoo 1h sets land side by side.

    Two defaults keep the grids readable, and both announce themselves:
      tfs=GRID_TFS  - 1m and 1h only. Once cellblock.4-9 have built every
                      timeframe there are 70 frames in the kernel, and a
                      70-panel grid is a wall, not a chart.
      min_days      - sets spanning less than this are dropped AND NAMED. After
                      cellblock.4 the nine Yahoo 1m bases are ~30 days each;
                      30 days is not evidence about anything.

    For deliberate multi-timeframe work, ask for it:
        collect_assets(tfs=TF_ORDER, min_days=0)
    Daily is absent from TF_ORDER on purpose - see the note there.
    """
    g, out, dropped = (scope if scope is not None else globals()), {}, []
    for symbol, var in VAR_OF.items():
        for tf in tfs:
            df = g.get(f"{var}_{tf}")
            if not (isinstance(df, pd.DataFrame) and len(df)
                    and isinstance(df.index, pd.DatetimeIndex)):
                continue
            df.attrs.setdefault("tf", tf)
            if df.index.tz is None:
                # yfinance hands back tz-naive stamps for DAILY bars. The
                # session rules are wall-clock, so a naive set cannot be placed
                # on a trading day - named here rather than crashing a grid.
                dropped.append(f"{symbol} {tf} (tz-naive)")
                continue
            days = (df.index[-1] - df.index[0]).days
            if days < min_days:
                dropped.append(f"{symbol} {tf} ({days}d)")
                continue
            out[f"{symbol} {tf}"] = (symbol, df)

    if not quiet:
        print(f"collected {len(out)} datasets across "
              f"{len({s for s, _ in out.values()})} assets")
        for lab, (sym, df) in out.items():
            yrs = (df.index[-1] - df.index[0]).days / 365.25
            print(f"  {lab:12} {len(df):>10,} bars  "
                  f"{df.index[0]:%Y-%m-%d} -> {df.index[-1]:%Y-%m-%d}  ({yrs:5.1f} yrs)"
                  + ("" if INSTRUMENTS[sym]["tradable"] else "   [not tradable]"))
        if dropped:
            print(f"  dropped ({min_days}d minimum, tz-aware only): "
                  f"{', '.join(dropped)}")
    return out


def _style(ax, t, ylabel=None, money=True, small=False):
    """Recessive chrome: hairline grid, muted ticks, no box, dollars on y."""
    ax.set_facecolor(t["surface"])
    ax.grid(True, color=t["grid"], linewidth=0.8, alpha=1.0)
    ax.set_axisbelow(True)
    for side, spine in ax.spines.items():
        spine.set_visible(side == "bottom")
        spine.set_color(t["axis"])
        spine.set_linewidth(1.0)
    ax.tick_params(colors=t["muted"], labelsize=7 if small else 8, length=0)
    if money:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(
            lambda v, _: (f"-${abs(v)/1000:,.0f}k" if abs(v) >= 1000 else f"-${abs(v):,.0f}")
            if v < 0 else (f"${v/1000:,.0f}k" if v >= 1000 else f"${v:,.0f}")))
    if ylabel:
        ax.set_ylabel(ylabel, color=t["secondary"], fontsize=9)
    return ax


def _equity_path(trades):
    """Equity stepped through time, anchored at 0 before the first trade."""
    x = [trades.index[0]] + list(pd.DatetimeIndex(trades["exit_ts"]))
    y = [0.0] + list(trades["equity"])
    return pd.DatetimeIndex(x), np.asarray(y, dtype=float)


def _span(df):
    """'2010-2026' - the panel's own period, since panels do not share an axis."""
    return f"{df.index[0]:%Y}-{df.index[-1]:%Y}"


def _blank_panel(ax, t, label, why):
    """A dataset that produced nothing still gets a panel, and a reason."""
    ax.set_facecolor(t["surface"])
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.add_patch(plt.Rectangle((0.02, 0.06), 0.96, 0.88, transform=ax.transAxes,
                               fill=False, edgecolor=t["grid"], linewidth=1.0,
                               linestyle=(0, (4, 4))))
    ax.text(0.5, 0.58, label, ha="center", va="center", fontsize=9.5,
            color=t["secondary"], transform=ax.transAxes)
    ax.text(0.5, 0.36, why, ha="center", va="center", fontsize=7.5,
            color=t["muted"], transform=ax.transAxes)


def _run_one(bars, symbol, account=None, seed=FLIP_SEED, **kw):
    """Run one dataset. Returns (trades, book, reason-it-produced-nothing)."""
    spec = INSTRUMENTS[symbol]
    if not spec["tradable"]:
        return None, None, f"{spec['unit']}\ncannot hold a unit"
    try:
        sig = coin_flip_signals(bars, symbol, tf=bars.attrs.get("tf"), seed=seed)
        tr, sk, bk = run_session_trades(bars, sig, symbol, account=account, **kw)
    except Exception as exc:
        # A grid draws many datasets. One malformed set must degrade to a
        # labelled panel, not take down every other panel with it - the reason
        # is printed in the panel, so nothing is swallowed.
        return None, None, f"{type(exc).__name__}\n{str(exc).splitlines()[0][:64]}"
    if len(tr) == 0:
        return None, None, (sk["reason"].mode()[0] if len(sk) else "no sessions")
    return tr, bk, None


def plot_pnl(trades, book, symbol, account=None, theme="light", figsize=(13, 7)):
    """Equity curve over drawdown, for one dataset. Returns the plotted frame.

    Two stacked panels on ONE shared time axis - never two y-scales on one plot.
    Top: cumulative net P&L. Bottom: drawdown from peak, against the account
    loss limit that ends the run.
    """
    a = {**ACCOUNT, **(account or {})}
    t = VIZ[theme]
    if trades is None or len(trades) == 0:
        print(f"{symbol}: no trades to plot")
        return trades

    x, eq = _equity_path(trades)
    dd    = np.maximum.accumulate(eq) - eq
    limit = a["loss_limit"]
    final = eq[-1]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, sharex=True, figsize=figsize, facecolor=t["surface"],
        gridspec_kw={"height_ratios": [2.6, 1], "hspace": 0.08})

    _style(ax1, t, "Cumulative net P&L ($)")
    ax1.axhline(0, color=t["axis"], linewidth=1.0)
    ax1.plot(x, eq, color=t["series"], linewidth=2.0, solid_joinstyle="round")
    ax1.fill_between(x, 0, eq, color=t["series"], alpha=0.10, linewidth=0)
    # one direct label, on the value that matters - not a number on every point
    ax1.plot([x[-1]], [final], "o", markersize=8, color=t["series"],
             markeredgecolor=t["surface"], markeredgewidth=2, zorder=5)
    ax1.annotate(f"  ${final:+,.0f}", (x[-1], final), color=t["ink"],
                 fontsize=10, fontweight="bold", va="center")
    ax1.set_title(
        f"{symbol}  -  cumulative net P&L, {len(trades):,} coin-flip trades"
        f"   |   {INSTRUMENTS[symbol]['unit']}, ${risk_per_trade(a):,.0f} risk/trade"
        f"   |   {x[0]:%Y-%m-%d} to {x[-1]:%Y-%m-%d}",
        color=t["ink"], fontsize=10.5, loc="left", pad=12)

    _style(ax2, t, "Drawdown ($)")
    ax2.fill_between(x, 0, -dd, color=t["loss"], alpha=0.18, linewidth=0)
    ax2.plot(x, -dd, color=t["loss"], linewidth=1.6)
    ax2.axhline(-limit, color=t["loss"], linewidth=1.2, linestyle=(0, (5, 4)))
    ax2.annotate(f"account loss limit  -${limit:,.0f}", (x[0], -limit),
                 color=t["loss"], fontsize=8.5, va="bottom", ha="left",
                 xytext=(4, 3), textcoords="offset points")
    ax2.set_ylim(min(-limit * 1.12, -dd.max() * 1.12), limit * 0.06)
    _wi   = int(np.argmax(dd))
    _frac = _wi / max(len(x) - 1, 1)
    _ha   = "right" if _frac > 0.72 else "left" if _frac < 0.28 else "center"
    _dx   = -6 if _ha == "right" else 6 if _ha == "left" else 0
    ax2.annotate(f"worst  -${dd.max():,.0f}  ({100 * dd.max() / limit:.0f}% of the limit)",
                 (x[_wi], -dd.max()), color=t["ink"], fontsize=8.5,
                 va="bottom", ha=_ha, xytext=(_dx, 8), textcoords="offset points")

    # kill switch gets a line AND a label - never colour alone
    if book is not None and book.killed:
        for ax in (ax1, ax2):
            ax.axvline(book.kill_ts, color=t["loss"], linewidth=1.2,
                       linestyle=(0, (2, 3)), zorder=1)
        ax1.annotate("  KILL SWITCH TRIPPED", (book.kill_ts, ax1.get_ylim()[1]),
                     color=t["loss"], fontsize=9, fontweight="bold", va="top")

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.autofmt_xdate()
    plt.show()
    out = pd.DataFrame({"equity": eq, "drawdown": dd}, index=x)
    out.index.name = "ts"
    return out


def _fan_paths(bars, symbol, seeds, base_seed, account, **kw):
    """Run `seeds` coins over one dataset. Returns (paths, deaths, rows)."""
    paths, deaths, rows = [], [], []
    for k in range(seeds):
        seed = base_seed + k
        tr, bk, _why = _run_one(bars, symbol, account=account, seed=seed, **kw)
        if tr is None:
            continue
        x, eq = _equity_path(tr)
        paths.append((x, eq))
        if bk.killed:
            deaths.append((x[-1], eq[-1]))
        rows.append(dict(seed=seed, trades=len(tr), net=eq[-1],
                         max_dd=float((np.maximum.accumulate(eq) - eq).max()),
                         killed=bk.killed))
    return paths, deaths, rows


def plot_pnl_fan(bars, symbol, seeds=COIN_SEEDS, base_seed=FLIP_SEED, account=None,
                 theme="light", figsize=(13, 6.5), **kw):
    """One dataset under `seeds` different coins. Returns the per-seed summary.

    This is the chart that says whether an equity curve means anything. Every
    grey line is the identical strategy on identical bars - only the coin
    differs. The width of that bundle is the honest uncertainty; the blue line
    is just the one draw you happened to look at first.
    """
    a = {**ACCOUNT, **(account or {})}
    t, limit = VIZ[theme], {**ACCOUNT, **(account or {})}["loss_limit"]
    paths, deaths, rows = _fan_paths(bars, symbol, seeds, base_seed, account, **kw)
    if not paths:
        print(f"{symbol}: no trades under any seed")
        return pd.DataFrame()

    summary = pd.DataFrame(rows).set_index("seed")
    fig, ax = plt.subplots(figsize=figsize, facecolor=t["surface"])
    _style(ax, t, "Cumulative net P&L ($)")
    ax.axhline(0, color=t["axis"], linewidth=1.0)
    for x, eq in paths:
        ax.plot(x, eq, color=t["ensemble"], linewidth=0.8, alpha=0.30, zorder=2)
    # the kill switch fires on DRAWDOWN FROM PEAK, not on an absolute balance,
    # so there is no horizontal line to draw for it - a run can die at +$90k.
    # Mark where each one actually ended instead.
    if deaths:
        ax.plot([d[0] for d in deaths], [d[1] for d in deaths], "x", markersize=7,
                markeredgewidth=1.6, color=t["loss"], zorder=6)
    x0, eq0 = paths[0]
    ax.plot(x0, eq0, color=t["series"], linewidth=2.2, zorder=4)
    ax.plot([x0[-1]], [eq0[-1]], "o", markersize=8, color=t["series"],
            markeredgecolor=t["surface"], markeredgewidth=2, zorder=5)

    # legend: identity is never carried by colour alone
    ax.plot([], [], color=t["ensemble"], linewidth=0.8, alpha=0.6,
            label=f"{len(paths)} coin seeds, identical rules")
    ax.plot([], [], color=t["series"], linewidth=2.2,
            label=f"seed {base_seed} (the one in the notebook)")
    if deaths:
        ax.plot([], [], "x", markersize=7, markeredgewidth=1.6, color=t["loss"],
                label=f"kill switch tripped ({len(deaths)} runs ended here)")
    leg = ax.legend(loc="upper left", frameon=False, fontsize=8.5)
    for txt in leg.get_texts():
        txt.set_color(t["secondary"])

    killed = int(summary["killed"].sum())
    ax.set_title(
        f"{symbol}  -  the same strategy under {len(paths)} different coins"
        f"   |   net ${summary['net'].min():+,.0f} to ${summary['net'].max():+,.0f}"
        f",  {killed}/{len(paths)} hit the ${limit:,.0f} drawdown limit",
        color=t["ink"], fontsize=10.5, loc="left", pad=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.autofmt_xdate()
    plt.show()
    return summary


def plot_pnl_grid(sets=None, account=None, theme="light", ncols=3,
                  panel=(4.5, 2.7), **kw):
    """ONE equity panel per dataset, every asset in the notebook. Returns a table.

    Ten-plus datasets on one axis would be cycled colours and an unreadable
    tangle, so this facets: one series per panel, no legend needed, every panel
    directly labelled with its own net and its own date span.
    """
    a    = {**ACCOUNT, **(account or {})}
    t    = VIZ[theme]
    sets = collect_assets(quiet=True) if sets is None else sets
    if not sets:
        print("no datasets found - run Module.2 first")
        return pd.DataFrame()

    nrows = math.ceil(len(sets) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel[0] * ncols, panel[1] * nrows),
                             facecolor=t["surface"], squeeze=False)
    rows = []
    for ax, (lab, (sym, bars)) in zip(axes.ravel(), sets.items()):
        tr, bk, why = _run_one(bars, sym, account=account, **kw)
        if tr is None:
            _blank_panel(ax, t, f"{lab}   no trades", why)
            rows.append(dict(dataset=lab, symbol=sym, span=_span(bars), trades=0,
                             net=np.nan, max_dd=np.nan, killed=False, note=why.replace("\n", " ")))
            continue
        x, eq = _equity_path(tr)
        dd    = float((np.maximum.accumulate(eq) - eq).max())
        _style(ax, t, small=True)
        ax.axhline(0, color=t["axis"], linewidth=1.0)
        col = t["series"] if eq[-1] >= 0 else t["loss"]
        ax.plot(x, eq, color=col, linewidth=1.6)
        ax.fill_between(x, 0, eq, color=col, alpha=0.10, linewidth=0)
        ax.set_title(f"{lab}  {_span(bars)}   ${eq[-1]:+,.0f}   {len(tr):,} trades"
                     + ("   KILLED" if bk.killed else ""),
                     color=t["loss"] if bk.killed else t["ink"],
                     fontsize=8.5, loc="left", pad=5)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        rows.append(dict(dataset=lab, symbol=sym, span=_span(bars), trades=len(tr),
                         net=eq[-1], max_dd=dd, killed=bk.killed, note=""))
    for ax in axes.ravel()[len(sets):]:
        ax.set_visible(False)
    fig.suptitle(f"Cumulative net P&L, every dataset in the notebook  -  1 unit each, "
                 f"${risk_per_trade(a):,.0f} risk/trade, one coin (seed {FLIP_SEED})"
                 f"   ·   panels do NOT share an x axis",
                 color=t["ink"], fontsize=11, x=0.006, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.975))
    plt.show()
    return pd.DataFrame(rows).set_index("dataset")


def plot_fan_grid(sets=None, seeds=COIN_SEEDS, base_seed=FLIP_SEED, account=None,
                  theme="light", ncols=3, panel=(4.5, 2.7), **kw):
    """COIN_SEEDS coins on EVERY dataset, one panel each. Returns a table.

    The grid version of the only chart worth trusting. Read the spread inside
    each panel before reading the level of any single line: if the bundle
    straddles zero, that dataset has told you nothing about edge.
    """
    a    = {**ACCOUNT, **(account or {})}
    t    = VIZ[theme]
    sets = collect_assets(quiet=True) if sets is None else sets
    if not sets:
        print("no datasets found - run Module.2 first")
        return pd.DataFrame()
    limit = a["loss_limit"]

    nrows = math.ceil(len(sets) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel[0] * ncols, panel[1] * nrows),
                             facecolor=t["surface"], squeeze=False)
    rows = []
    for ax, (lab, (sym, bars)) in zip(axes.ravel(), sets.items()):
        print(f"  {lab:12} {seeds} coins ...", end="", flush=True)
        paths, deaths, seed_rows = _fan_paths(bars, sym, seeds, base_seed, account, **kw)
        if not paths:
            _, _, why = _run_one(bars, sym, account=account, **kw)
            print(f" none ({why})".replace("\n", " "))
            _blank_panel(ax, t, f"{lab}   no trades", why or "no trades")
            rows.append(dict(dataset=lab, symbol=sym, span=_span(bars), seeds=0,
                             net_min=np.nan, net_med=np.nan, net_max=np.nan,
                             pct_profitable=np.nan, killed=0,
                             note=(why or "").replace("\n", " ")))
            continue
        s = pd.DataFrame(seed_rows)
        print(f" median ${s['net'].median():+,.0f}, {int(s['killed'].sum())} killed")

        _style(ax, t, small=True)
        ax.axhline(0, color=t["axis"], linewidth=1.0)
        for x, eq in paths:
            ax.plot(x, eq, color=t["ensemble"], linewidth=0.7, alpha=0.28, zorder=2)
        if deaths:
            ax.plot([d[0] for d in deaths], [d[1] for d in deaths], "x", markersize=5,
                    markeredgewidth=1.2, color=t["loss"], zorder=6)
        ax.plot(paths[0][0], paths[0][1], color=t["series"], linewidth=1.7, zorder=4)
        killed = int(s["killed"].sum())
        ax.set_title(f"{lab}  {_span(bars)}   median ${s['net'].median():+,.0f}"
                     f"   {killed}/{len(paths)} killed"
                     f"   {100 * (s['net'] > 0).mean():.0f}% profitable",
                     color=t["loss"] if killed > len(paths) / 2 else t["ink"],
                     fontsize=8.5, loc="left", pad=5)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        rows.append(dict(dataset=lab, symbol=sym, span=_span(bars), seeds=len(paths),
                         net_min=s["net"].min(), net_med=s["net"].median(),
                         net_max=s["net"].max(),
                         pct_profitable=100 * (s["net"] > 0).mean(),
                         killed=killed, note=""))
    for ax in axes.ravel()[len(sets):]:
        ax.set_visible(False)

    # one figure-level legend - per-panel legends would be pure noise here
    h = [plt.Line2D([], [], color=t["ensemble"], lw=0.9, alpha=0.6),
         plt.Line2D([], [], color=t["series"], lw=1.7),
         plt.Line2D([], [], color=t["loss"], marker="x", lw=0, markersize=6,
                    markeredgewidth=1.4)]
    leg = fig.legend(h, [f"{seeds} coin seeds, identical rules",
                         f"seed {base_seed}",
                         f"kill switch tripped (${limit:,.0f} drawdown)"],
                     loc="upper right", frameon=False, fontsize=8.5, ncols=3,
                     bbox_to_anchor=(0.995, 1.0))
    for txt in leg.get_texts():
        txt.set_color(t["secondary"])
    fig.suptitle(f"The same coin-flip strategy under {seeds} coins, every dataset"
                 f"  -  1 unit each, ${risk_per_trade(a):,.0f} risk/trade"
                 f"   ·   panels do NOT share an x axis",
                 color=t["ink"], fontsize=11, x=0.006, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.972))
    plt.show()
    return pd.DataFrame(rows).set_index("dataset")


def check_pnl_plots(days=120, symbol="/NQ", seeds=COIN_SEEDS):
    """Self-test: the drawn numbers must equal the traded numbers."""
    tz  = SESSION["tz"]
    idx = pd.date_range("2024-01-01 00:00", periods=days * 24, freq="60min", tz=tz)
    idx = idx[idx.dayofweek < 5]
    rng = np.random.default_rng(11)
    px  = 18_000 * np.exp(np.cumsum(rng.normal(0, 1.5e-3, len(idx))))
    rad = np.abs(rng.normal(0, 12, len(idx))) + 3
    bars = pd.DataFrame({"open": px, "high": px + rad, "low": px - rad,
                         "close": px + rng.normal(0, 6, len(idx)),
                         "volume": rng.integers(1, 9999, len(idx))}, index=idx)
    bars["high"] = bars[["open", "high", "close"]].max(axis=1)
    bars["low"]  = bars[["open", "low",  "close"]].min(axis=1)
    bars.attrs["tf"] = "60m"

    sig = coin_flip_signals(bars, symbol, tf="60m")
    tr, sk, bk = run_session_trades(bars, sig, symbol)
    curve = plot_pnl(tr, bk, symbol)
    checks = []

    # 1. the curve's endpoint IS the sum of the trades - no cosmetic smoothing
    checks.append(("equity endpoint == sum of net P&L",
                   abs(curve["equity"].iloc[-1] - tr["net"].sum()) < 1e-6))

    # 2. the curve starts flat at zero, before any trade has closed
    checks.append(("curve anchored at 0 before the first trade",
                   abs(curve["equity"].iloc[0]) < 1e-12
                   and curve.index[0] == tr.index[0]))

    # 3. the drawdown panel matches the drawdown the risk book measured
    checks.append(("plotted drawdown == peak-to-trough of the equity",
                   abs(curve["drawdown"].max()
                       - (tr["equity"].cummax() - tr["equity"]).max()) < 1e-6))

    # 4. every point is monotone in time - nothing is drawn out of order
    checks.append(("time axis strictly non-decreasing",
                   bool(curve.index.is_monotonic_increasing)))

    # 5. the fan really varies the coin over all COIN_SEEDS draws
    fan = plot_pnl_fan(bars, symbol, seeds=seeds)
    checks.append((f"fan draws {seeds} distinct coins with a real spread",
                   len(fan) == seeds and fan["net"].nunique() == seeds))

    # 6. a grid must not silently drop a dataset it cannot trade
    grid = plot_pnl_grid({"SELFTEST 60m": (symbol, bars),
                          "VIX 60m": ("VIX", bars)}, ncols=2)
    checks.append(("untradable dataset still gets a row, with a reason",
                   len(grid) == 2 and grid.loc["VIX 60m", "trades"] == 0
                   and bool(grid.loc["VIX 60m", "note"])))

    print(f"P&L chart self-test  ({len(tr):,} trades, {seeds} coins)")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise AssertionError(f"P&L chart self-test FAILED: {failed}")
    print("  the charts show the trades, not a prettier version of them")
    return curve


_ = check_pnl_plots()

# Usage, once Module.2 has loaded the data:
#   sets = collect_assets()          # /NQ 1m local + every {asset}_1h
#   plot_pnl_grid(sets)              # one coin,  every dataset
#   plot_fan_grid(sets)              # 50 coins,  every dataset   <- read this one
#
#   tr, bk, _ = _run_one(nq_1m, "/NQ")
#   plot_pnl(tr, bk, "/NQ")          # one dataset, full detail


In [ ]:
# cellblock.4

# data layer  +  the 1m bases every resampled set below is aggregated from.
#
# The resampling engine and the fetchers live HERE because the cells that used
# to hold them - Module.3 cellblock.1 and cellblock.2 - are now the strategy and
# the risk layer. cellblocks 5 to 10 below call into what this cellblock
# defines, so this one has to run before any of them.
#
#   /NQ    : LOCAL continuous front-month from Module.2 cellblock.1
#            (5.3M bars, 2010-2026) - NOT Yahoo's 30-day window.
#   others : yfinance 1m, fetched in chunks, reaches back only ~30 days.
#            Anything resampled from those inherits that 30-day ceiling.
#
# It also builds {asset}_1h over YF_PERIOD (720d) for ALL ten assets - the only
# set with real history for TSLA, NVDA, AAPL and VIX, and the one the strategy
# actually trades.
import time
import logging
from contextlib import contextmanager

AGG = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}

# cellblock.1 already defines TIMEFRAMES. Repeated identically so this cellblock
# still stands up if the strategy cells have not been run in this kernel.
TIMEFRAMES = {"1m": "1min", "3m": "3min", "5m": "5min",
              "15m": "15min", "30m": "30min", "60m": "60min"}

ASSETS = [("nq",   "/NQ",  "NQ=F"),
          ("es",   "/ES",  "ES=F"),
          ("rty",  "/RTY", "RTY=F"),
          ("ym",   "/YM",  "YM=F"),
          ("btc",  "BTC",  "BTC-USD"),
          ("eth",  "ETH",  "ETH-USD"),
          ("tsla", "TSLA", "TSLA"),
          ("nvda", "NVDA", "NVDA"),
          ("aapl", "AAPL", "AAPL"),
          ("vix",  "VIX",  "^VIX"),
          ("vixy", "VIXY", "VIXY")]     # tradable proxy for the VIX index

RESAMPLED = ["3m", "5m", "15m", "30m", "60m"]


def resample_ohlcv(df, tf, src_step=None, drop_partial=True, label=""):
    """Aggregate an OHLCV frame up to timeframe `tf`.

    Bars are left-labelled/left-closed, so a bar is stamped at its OPEN and only
    becomes complete one `tf` later. Empty bins (weekends, halts, session breaks)
    are dropped rather than forward-filled - a forward-filled bar is a bar that
    never traded. Upsampling is refused outright.

    The refusal MEASURES the source step off the index rather than believing
    `src_step`. It used to trust the argument, and every batch cellblock below
    passes the same constant: hand a 1h frame to resample_ohlcv(df, "15m") with
    src_step="1min" and the guard saw 15min > 1min, raised nothing, and returned
    hourly bars wearing a 15m label - one source bar per bin. Silent upsampling
    is exactly the failure this function exists to prevent, so the measurement
    wins, and a declared step can only make the check stricter, never looser.
    """
    freq = TIMEFRAMES.get(tf, tf)
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{label}: index must be a DatetimeIndex, got {type(df.index).__name__}")
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()

    measured = (pd.Timedelta(df.index.to_series().diff().median())
                if len(df) >= 3 else None)
    declared = pd.Timedelta(src_step) if src_step is not None else None
    src = max([s for s in (measured, declared) if s is not None and s > pd.Timedelta(0)],
              default=pd.Timedelta("1min"))
    if pd.Timedelta(freq) < src:
        raise ValueError(
            f"{label}: refusing to resample a {src} source UP to {tf}. "
            "Upsampling fabricates bars that never traded."
            + ("" if declared is None or declared == src else
               f" (caller declared {declared}; the index says {measured})"))

    g     = df.resample(freq, label="left", closed="left")
    out   = g.agg(AGG)
    n_src = g.size().reindex(out.index)
    out, n_src = out[n_src > 0], n_src[n_src > 0]        # drop empty bins

    # a trailing bin the source never reached is incomplete -> its "close" is fake
    dropped_partial = False
    if drop_partial and len(out):
        if df.index[-1] + src < out.index[-1] + pd.Timedelta(freq):
            out, n_src, dropped_partial = out.iloc[:-1], n_src.iloc[:-1], True

    out["volume"] = out["volume"].fillna(0)
    out.index.name = "ts"
    # store the MEASURED step, not the declared one - describe_bars divides by
    # this, and src_step now defaults to None
    out.attrs.update(tf=tf, src_step=src, label=label,
                     n_src=n_src, dropped_partial=dropped_partial)
    return out


def describe_bars(df, label, tf):
    """Full coverage + integrity report for one set. Returns it unchanged."""
    n_src   = df.attrs.get("n_src")
    span    = (df.index[-1] - df.index[0]).total_seconds() / 86400
    bad_rng = int((~((df["high"] >= df[["open", "close"]].max(axis=1) - 1e-6) &
                     (df["low"]  <= df[["open", "close"]].min(axis=1) + 1e-6))).sum())
    print(f"{label}  {tf}")
    print(f"  bars      : {len(df):,}")
    print(f"  coverage  : {df.index[0]}  ->  {df.index[-1]}   ({span:,.1f} days)")
    print(f"  close     : {df['close'].min():,.2f} - {df['close'].max():,.2f}")
    print(f"  volume    : {df['volume'].sum():,.0f}"
          + ("   [!] all-zero: source reports no volume" if df["volume"].sum() == 0 else ""))
    print(f"  integrity : high<low {int((df['high'] < df['low']).sum())} | "
          f"OHLC out of range {bad_rng} | NaN {int(df.isna().sum().sum())} | "
          f"dup ts {int(df.index.duplicated().sum())}")
    if n_src is not None and len(n_src):
        full = int(pd.Timedelta(TIMEFRAMES[tf]) / pd.Timedelta(df.attrs["src_step"]))
        print(f"  bin fill  : median {n_src.median():.0f}/{full} source bars, "
              f"min {n_src.min():.0f}, thin(<50%) {int((n_src < full * 0.5).sum()):,}")
    if df.attrs.get("dropped_partial"):
        print(f"  [note] trailing incomplete {tf} bar dropped")
    return df


def summarise(df, label, tf):
    """One compact line per asset, for the batch cellblocks below."""
    flags = []
    if (df["high"] < df["low"]).any():               flags.append("high<low")
    if df.index.duplicated().any():                  flags.append("dup-ts")
    if df[["open", "high", "low", "close"]].isna().any().any(): flags.append("NaN")
    if df["volume"].sum() == 0:                      flags.append("no-volume")
    if df.attrs.get("dropped_partial"):              flags.append("partial-dropped")
    print(f"  {label:6}{tf:>5}  {len(df):>9,} bars   "
          f"{df.index[0]:%Y-%m-%d %H:%M} -> {df.index[-1]:%Y-%m-%d %H:%M}   "
          f"{'| ' + ', '.join(flags) if flags else 'clean'}")
    return df


def _clean(raw, label, ticker):
    """Flatten yfinance output to a sorted, deduped, tz-aware OHLCV frame."""
    if raw is None or len(raw) == 0:
        raise RuntimeError(f"{label} ({ticker}): yfinance returned no rows")
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(-1)
    df = raw.rename(columns=str.lower)
    missing = [c for c in OHLCV if c not in df.columns]
    if missing:
        raise RuntimeError(f"{label} ({ticker}): missing columns {missing}")
    df = df[OHLCV].copy()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df.dropna(subset=["open", "high", "low", "close"])
    df["volume"] = df["volume"].fillna(0)
    df.index.name = "ts"
    df.columns.name = None
    return df


def fetch_daily(ticker, label):
    """Maximum available daily history. This is the only set with real depth."""
    df = _clean(yf.download(ticker, period="max", interval="1d", auto_adjust=False,
                            progress=False, threads=False), label, ticker)
    print(f"  {label:6}  1d  {len(df):>9,} bars  "
          f"{df.index[0]:%Y-%m-%d} -> {df.index[-1]:%Y-%m-%d}  "
          f"({(df.index[-1] - df.index[0]).days / 365.25:5.1f} yrs)")

    # Yahoo's daily futures bars contain bars whose open/close sit OUTSIDE the
    # high/low range. Reported, never silently repaired - a backtest that assumes
    # low <= open,close <= high would compute impossible fills on these dates.
    bad = ~((df["high"] >= df[["open", "close"]].max(axis=1) - 1e-6) &
            (df["low"]  <= df[["open", "close"]].min(axis=1) + 1e-6))
    if bad.any():
        print(f"          [!] {int(bad.sum())} bar(s) with OHLC outside the high/low "
              f"range ({100 * bad.mean():.2f}%) - source defect, NOT repaired. "
              f"Last: {df.index[bad][-1]:%Y-%m-%d}.")
    return df


YF_1M_LOOKBACK_DAYS = 30      # Yahoo keeps roughly this much 1m history
YF_1M_CHUNK_DAYS    = 7       # and hands back at most 8 days per request


@contextmanager
def _quiet_yf():
    """Mute yfinance's own logger for the duration of a request.

    It logs "$TSLA: possibly delisted; no price data found" whenever a request
    window contains no session at all. The usual cause is mundane: the oldest
    chunk of the walk lands on a Saturday/Sunday, and an RTH-only symbol has no
    minutes there. TSLA is not delisted. Empty windows are counted and reported
    by fetch_1m itself, so nothing is hidden - it is just not shouted. A symbol
    that really has no data still fails loudly, because every window comes back
    empty and the RuntimeError below fires.
    """
    lg   = logging.getLogger("yfinance")
    prev = lg.level
    lg.setLevel(logging.CRITICAL)
    try:
        yield
    finally:
        lg.setLevel(prev)


def fetch_1m(ticker, label, lookback=YF_1M_LOOKBACK_DAYS,
             chunk=YF_1M_CHUNK_DAYS, pause=0.4):
    """Walk Yahoo's 1m window in chunks, since it caps each request at 8 days."""
    end         = pd.Timestamp.utcnow().normalize() + pd.Timedelta(days=1)
    start_floor = end - pd.Timedelta(days=lookback)
    parts, empty, fails, win_end = [], [], 0, end

    while win_end > start_floor:
        win_start = max(win_end - pd.Timedelta(days=chunk), start_floor)
        try:
            with _quiet_yf():
                raw = yf.download(ticker, start=win_start.date(), end=win_end.date(),
                                  interval="1m", auto_adjust=False, progress=False,
                                  threads=False)
            if raw is not None and len(raw):
                parts.append(_clean(raw, label, ticker))
            else:
                empty.append((win_start, win_end))
        except Exception:
            fails += 1                       # a real request error, not an empty window
        win_end = win_start
        time.sleep(pause)

    if not parts:
        raise RuntimeError(f"{label} ({ticker}): no 1m data in the last {lookback}d "
                           f"({len(empty)} empty window(s), {fails} error(s)) - "
                           "this one really does have no data")
    df = pd.concat(parts).sort_index()
    df = df[~df.index.duplicated(keep="last")]

    # yfinance's `end` is exclusive, so a window is "all weekend" when no
    # business day falls in [start, end).
    closed = sum(1 for a, b in empty
                 if len(pd.bdate_range(a.date(), b.date() - pd.Timedelta(days=1))) == 0)
    note = ""
    if empty:
        note += f"   [{len(empty)} empty window(s)"
        note += f", {closed} weekend/holiday]" if closed else "]"
    if fails:
        note += f"   [{fails} request error(s)]"
    print(f"  {label:6}  1m  {len(df):>9,} bars  "
          f"{df.index[0]:%Y-%m-%d %H:%M} -> {df.index[-1]:%Y-%m-%d %H:%M}" + note)
    return df


# ---- {asset}_1d  -  daily, maximum history. Deepest data in the system -------
# Kept for regime work and long-horizon checks. NOT usable by the cellblock.2
# entry rule: that needs a bar inside the hour before the cash open, and a daily
# bar has no inside. The chart grids exclude 1d for exactly that reason.
print("daily (period='max')")
for _var, _lab, _tkr in ASSETS:
    globals()[f"{_var}_1d"] = fetch_daily(_tkr, _lab)

# ---- {asset}_1h  -  720 days of hourly bars, every asset ---------------------
# This is the set the strategy actually trades: long enough to mean something
# (~2 yrs) and available for ALL ten, including TSLA/NVDA/AAPL/VIX, whose 1m
# window is only ~30 days. Module.2 cellblocks 2-11 build the same frames one
# at a time with a chart each; this rebuilds them in one pass so the data layer
# stands alone - if those cells were skipped, the grids would silently show
# only the assets that happened to be loaded.
print(f"\n1h (period={YF_PERIOD}, interval={YF_INTERVAL})")
for _var, _lab, _tkr in ASSETS:
    _df = fetch_1h(_tkr, _lab)
    globals()[f"{_var}_1h"] = _df
    _zero = "   [!] no volume reported" if _df["volume"].sum() == 0 else ""
    print(f"  {_lab:6}  1h  {len(_df):>9,} bars  "
          f"{_df.index[0]:%Y-%m-%d %H:%M} -> {_df.index[-1]:%Y-%m-%d %H:%M}  "
          f"({(_df.index[-1] - _df.index[0]).days:>4d} days){_zero}")

# ---- {asset}_1m  -  the bases cellblocks 5-9 aggregate up from ---------------
if "nq_1m" not in globals():
    raise RuntimeError("run Module.2 cellblock.1 first - it builds nq_1m from the local CSV")

print("\n1m bases")
print(f"  {'/NQ':6}  1m  {len(nq_1m):>9,} bars  "
      f"{nq_1m.index[0]:%Y-%m-%d %H:%M} -> {nq_1m.index[-1]:%Y-%m-%d %H:%M}"
      f"   [local CSV, {(nq_1m.index[-1] - nq_1m.index[0]).days / 365.25:.1f} yrs]")

for _var, _lab, _tkr in ASSETS:
    if _var == "nq":
        continue                                   # already built from the local CSV
    globals()[f"{_var}_1m"] = fetch_1m(_tkr, _lab)

print(f"\nbuilt: {', '.join(f'{v}_1d' for v, _, _ in ASSETS)}")
print(f"built: {', '.join(f'{v}_1h' for v, _, _ in ASSETS)}")
print(f"built: {', '.join(f'{v}_1m' for v, _, _ in ASSETS)}")


In [ ]:
# cellblock.5

# {asset}_3m  -  resampled from each asset's 1m base.
# 3m exists ONLY here - Yahoo does not serve a 3m interval at any period.
print("3m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "3m", label=_lab)
    globals()[f"{_var}_3m"] = summarise(_out, _lab, "3m")

print(f"\nbuilt: {', '.join(f'{v}_3m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_3m, "<label>", "3m") for the full report on any one set


In [ ]:
# cellblock.6

# {asset}_5m  -  resampled from each asset's 1m base.
print("5m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "5m", label=_lab)
    globals()[f"{_var}_5m"] = summarise(_out, _lab, "5m")

print(f"\nbuilt: {', '.join(f'{v}_5m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_5m, "<label>", "5m") for the full report on any one set


In [ ]:
# cellblock.7

# {asset}_15m  -  resampled from each asset's 1m base.
# NOTE: this supersedes Module.2's native 15m pull. For /NQ that is 366k bars
# back to 2010 instead of Yahoo's 60-day window.
print("15m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "15m", label=_lab)
    globals()[f"{_var}_15m"] = summarise(_out, _lab, "15m")

print(f"\nbuilt: {', '.join(f'{v}_15m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_15m, "<label>", "15m") for the full report on any one set


In [ ]:
# cellblock.8

# {asset}_30m  -  resampled from each asset's 1m base.
print("30m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "30m", label=_lab)
    globals()[f"{_var}_30m"] = summarise(_out, _lab, "30m")

print(f"\nbuilt: {', '.join(f'{v}_30m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_30m, "<label>", "30m") for the full report on any one set


In [ ]:
# cellblock.9

# {asset}_60m  -  resampled from each asset's 1m base.
print("60m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "60m", label=_lab)
    globals()[f"{_var}_60m"] = summarise(_out, _lab, "60m")

print(f"\nbuilt: {', '.join(f'{v}_60m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_60m, "<label>", "60m") for the full report on any one set


In [ ]:
# cellblock.10

# DATASETS registry + integrity / conservation verification
TFS = ["1d", "1m", "3m", "5m", "15m", "30m", "60m"]

DATASETS = {lab: {tf: globals()[f"{var}_{tf}"]
                  for tf in TFS if f"{var}_{tf}" in globals()}
            for var, lab, _ in ASSETS}

print("bar counts")
print(f"  {'asset':6}" + "".join(f"{tf:>11}" for tf in TFS))
print("  " + "-" * (6 + 11 * len(TFS)))
for _lab, _sets in DATASETS.items():
    print(f"  {_lab:6}" + "".join(
        f"{len(_sets[tf]):>11,}" if tf in _sets else f"{'-':>11}" for tf in TFS))

# ---- integrity -------------------------------------------------------------
problems = []
for _lab, _sets in DATASETS.items():
    for _tf, _df in _sets.items():
        if len(_df) == 0:
            problems.append(f"{_lab} {_tf}: EMPTY")
        if _df.index.duplicated().any():
            problems.append(f"{_lab} {_tf}: duplicate timestamps")
        if not _df.index.is_monotonic_increasing:
            problems.append(f"{_lab} {_tf}: not sorted")
        if (_df["high"] < _df["low"]).any():
            problems.append(f"{_lab} {_tf}: high < low")
        if _df[["open", "high", "low", "close"]].isna().any().any():
            problems.append(f"{_lab} {_tf}: NaN in OHLC")
        if list(_df.columns)[:5] != OHLCV:
            problems.append(f"{_lab} {_tf}: unexpected columns {list(_df.columns)}")
        # every resampled bar must start on the timeframe grid
        if _tf in TIMEFRAMES and _tf != "1m" and len(_df) > 2:
            _step = pd.Timedelta(TIMEFRAMES[_tf])
            if ((_df.index.to_series().diff().dropna() % _step)
                    != pd.Timedelta(0)).any():
                problems.append(f"{_lab} {_tf}: bars not aligned to the {_tf} grid")

print("\nintegrity:", "NONE" if not problems else "")
for p in problems:
    print(f"  [!] {p}")

# ---- volume conservation: aggregation must not create or destroy contracts --
print("\nvolume conservation (1m -> each resampled set):")
for _lab, _sets in DATASETS.items():
    if "1m" not in _sets:
        continue
    _base = _sets["1m"]["volume"].sum()
    _bad = [tf for tf in RESAMPLED
            if tf in _sets and not _sets[tf].attrs.get("dropped_partial")
            and _sets[tf]["volume"].sum() != _base]
    print(f"  {_lab:6} {'OK' if not _bad else 'FAIL -> ' + str(_bad)}"
          f"   (sets with a dropped partial bar are exempt)")

# ---- no-lookahead: a bar may only contain 1m bars from inside its own window -
print("\nno-lookahead spot check (25 random 15m bars per asset):")
_rng = np.random.default_rng(0)
for _lab, _sets in DATASETS.items():
    if "15m" not in _sets or "1m" not in _sets:
        continue
    _b, _r = _sets["1m"], _sets["15m"]
    _ok = True
    for _p in _rng.choice(len(_r), size=min(25, len(_r)), replace=False):
        _t = _r.index[_p]
        _w = _b.loc[_t : _t + pd.Timedelta("15min") - pd.Timedelta("1s")]
        if len(_w) == 0:
            continue
        if not (abs(_r["open"].iloc[_p]  - _w["open"].iloc[0])  < 1e-3 and
                abs(_r["close"].iloc[_p] - _w["close"].iloc[-1]) < 1e-3 and
                abs(_r["high"].iloc[_p]  - _w["high"].max())     < 1e-3 and
                abs(_r["low"].iloc[_p]   - _w["low"].min())      < 1e-3 and
                _r["volume"].iloc[_p] == _w["volume"].sum()):
            _ok = False
            break
    print(f"  {_lab:6} {'OK - every bar built only from 1m bars inside its window'
                        if _ok else 'FAIL - bar contains data from outside its window'}")


## Module.4
### Strategy importing

In [ ]:
# cellblock.1

# feature engineering  -  a time-series regime filter over the coin flip
#     trend_state()   : is the market trending up, down, or neither, right now
#     gate_signals()  : keep the flip only when it AGREES with that regime
#     compare_gate()  : the benchmark that says whether the coin still matters
#
#   bull + heads -> LONG        bull + tails -> no trade
#   bear + tails -> SHORT       bear + heads -> no trade
#   neither      -> no trade, whichever way the coin lands
#
# Read this before reading any result the filter produces:
#
#   The gate can only REMOVE trades, never reverse one. So every trade that
#   survives it is in the direction of the trend, and the coin no longer decides
#   direction at all - it decides, at random, which half of the trend signals to
#   skip. What you are testing after this cellblock is trend following, sampled
#   by a coin. compare_gate() therefore runs a third arm, "trend only", which
#   takes every trend signal and ignores the coin entirely. If gated and
#   trend-only have the same expectancy per trade, the coin is contributing
#   nothing but variance, and the honest conclusion is to drop it.
#
#   A filter also cannot create edge out of a fair coin on its own. Raw flips
#   have zero expectancy before costs; keeping the subset that points with the
#   trend earns whatever trend following earns on that market over that period,
#   no more. If the gated arm looks good, the question to ask is whether TREND
#   FOLLOWING worked in this sample - not whether the filter is clever.
import numpy as _np_check     # noqa: F401  (fails early if Module.1 not run)

# The regime is fitted on DAILY closes, not on bars, and the lookback is in
# TRADING DAYS. A bar count cannot mean the same thing twice here: 24 bars is
# one session of /NQ 1h, four days of AAPL 1h, and twenty minutes of /NQ 1m. A
# day is a day for all eleven assets, so the same number describes the same
# horizon everywhere, and "multi-day regime" is what it literally says.
TREND = {
    "method":     "slope_t",  # "slope_t" (regression) | "ema" (crossover)
    "lookback_days":     40,  # DAYS of daily closes the fit sees (~2 months)
    "t_min":            1.5,  # |t| below this is NOT a regime -> no trade
    "fast_days":         20,  # method="ema" only
    "slow_days":         80,  # method="ema" only
    "ema_min":       0.0015,  # method="ema": |fast-slow| in logs below this = neutral
    "day_cutoff":   "16:00",  # a day's close = its last bar at or before this, ET
}

BULL, BEAR, FLAT = 1, -1, 0


def _rolling_slope_t(y, n):
    """t-statistic of the OLS slope of `y` on 0..n-1, over a rolling window.

    Closed form rather than rolling(...).apply(), which would take minutes on a
    5M-bar series. x is always 0..n-1, so Sxx is a constant and the only moving
    parts are rolling sums of y, y^2 and j*y.

    The window ENDS at the bar being labelled, so the value at bar t uses bars
    t-n+1 .. t inclusive. trend_state() shifts it afterwards - see there.
    """
    y   = y.astype("float64")
    j   = np.arange(len(y), dtype="float64")
    sy  = y.rolling(n).sum()
    sjy = (pd.Series(j, index=y.index) * y).rolling(n).sum()
    syy = (y * y).rolling(n).sum()

    start = pd.Series(j - n + 1.0, index=y.index)
    xbar  = (n - 1) / 2.0
    Sxy   = (sjy - start * sy) - xbar * sy          # sum((x - xbar) * y)
    Sxx   = n * (n * n - 1.0) / 12.0                # sum((x - xbar)^2), constant
    Syy   = syy - sy * sy / n

    b   = Sxy / Sxx
    sse = (Syy - b * Sxy).clip(lower=0.0)           # float noise can go negative
    se  = np.sqrt((sse / (n - 2)) / Sxx)
    return (b / se.replace(0.0, np.nan)), b


def session_closes(bars, tz=None, cutoff=None):
    """One close per trading day: the last bar at or before `cutoff`, in ET.

    Anchored to the cash close rather than to the last bar of the calendar day,
    because that is the price the strategy itself exits at - so the regime is
    fitted on the same quantity the trades capture. Vectorised on the index's
    hour/minute rather than per-timestamp .time() calls, which matters at 5M bars.

    Returns a Series indexed by that day's ET midnight, so it can be mapped
    straight back onto intraday bars.
    """
    tz     = tz or SESSION["tz"]
    cutoff = cutoff or TREND["day_cutoff"]
    hh, mm = (int(x) for x in str(cutoff).split(":"))
    et     = bars.index.tz_convert(tz)
    keep   = (et.hour * 60 + et.minute) <= (hh * 60 + mm)
    if not keep.any():
        raise ValueError(f"no bars at or before {cutoff} {tz} - cannot form a daily close")
    day = et.normalize()
    return bars["close"].astype("float64")[keep].groupby(day[keep]).last()


def trend_state(bars, label="", method=None, lookback_days=None, t_min=None,
                fast_days=None, slow_days=None, ema_min=None, day_cutoff=None):
    """Regime per bar, usable AT that bar's open. Returns a frame on bars.index.

        trend       +1 bull / -1 bear / 0 neither
        strength    the t-statistic (slope_t) or the fast/slow gap (ema)
        raw_trend   the same regime BEFORE shifting - diagnostics only

    Causality, which is the whole game here. The estimate at bar t is computed
    from a window ENDING at t, so it contains bar t's own close - a price that
    is not known when bar t OPENS, and the open is when cellblock.2 enters.
    Every column is therefore shifted one bar before being returned, exactly as
    cellblock.1 shifts `signal` into `position` and cellblock.2 shifts ATR. The
    first `lookback` bars are NaN -> FLAT -> no trade, which is correct: there
    is no regime estimate yet, so there is nothing to agree with.
    """
    cfg = {**TREND}
    for k, v in dict(method=method, lookback_days=lookback_days, t_min=t_min,
                     fast_days=fast_days, slow_days=slow_days, ema_min=ema_min,
                     day_cutoff=day_cutoff).items():
        if v is not None:
            cfg[k] = v
    if not isinstance(bars.index, pd.DatetimeIndex):
        raise TypeError(f"{label}: index must be a DatetimeIndex")
    if not bars.index.is_monotonic_increasing:
        raise ValueError(f"{label}: index must be sorted ascending")
    if bars.index.tz is None:
        raise ValueError(f"{label}: bar index is tz-naive - a trading day has no "
                         "meaning without a zone. Localize it first.")

    daily = session_closes(bars, cutoff=cfg["day_cutoff"])
    logpx = np.log(daily)

    if cfg["method"] == "slope_t":
        n = int(cfg["lookback_days"])
        if n < 4:
            raise ValueError("lookback_days must be >= 4 for a slope t-statistic")
        strength, _slope = _rolling_slope_t(logpx, n)
        thr = float(cfg["t_min"])
    elif cfg["method"] == "ema":
        f = logpx.ewm(span=int(cfg["fast_days"]), adjust=False).mean()
        s = logpx.ewm(span=int(cfg["slow_days"]), adjust=False).mean()
        strength = f - s                       # log gap ~ fractional gap
        strength.iloc[:int(cfg["slow_days"])] = np.nan     # not warmed up
        thr = float(cfg["ema_min"])
    else:
        raise ValueError(f"unknown method {cfg['method']!r} - use 'slope_t' or 'ema'")

    raw_day = pd.Series(FLAT, index=daily.index, dtype="int8")
    raw_day[strength > thr] = BULL
    raw_day[strength < -thr] = BEAR
    raw_day[strength.isna()] = FLAT

    # Shift by one DAY, not one bar. The fit for day D ends at D's own close,
    # which is not known when D opens - and the entry happens before D's open.
    # After the shift, day D carries the regime measured through D-1's close,
    # which is the last thing that had finished happening. Every bar inside D
    # then gets that same value, so the regime is constant within a session and
    # no intraday price can feed back into it.
    use_day  = raw_day.shift(1).fillna(FLAT).astype("int8")
    str_day  = strength.shift(1)

    day_of = bars.index.tz_convert(SESSION["tz"]).normalize()
    out = pd.DataFrame(index=bars.index.copy())
    out["trend"]     = use_day.reindex(day_of).fillna(FLAT).to_numpy().astype("int8")
    out["strength"]  = str_day.reindex(day_of).to_numpy()
    out["raw_trend"] = raw_day.reindex(day_of).fillna(FLAT).to_numpy().astype("int8")
    out.index.name = "ts"
    out.attrs.update(label=label, n_days=int(len(daily)), **cfg)
    return out


def gate_signals(sig, ts, label=""):
    """Keep the flip only where it agrees with the regime. Returns a new frame.

    `position` is the only column cellblock.2 reads, and 0 means "no trade" -
    run_session_trades already skips a session whose position is 0. Nothing
    downstream needs to change. (Its skip reason will read "no tradable flip
    yet", which is now slightly off: the flip existed, the regime vetoed it.)

    The gate is strictly subtractive. A trade is taken only if
    position == trend, so a surviving trade always points with the trend and
    the gate can never produce a side the coin did not already pick.
    """
    if not sig.index.equals(ts.index):
        raise ValueError("signals and trend must share one index")
    pos   = sig["position"]
    agree = (pos == ts["trend"]) & (ts["trend"] != FLAT) & pos.notna()

    out = sig.copy()
    out["coin"]      = pos                                   # what the coin wanted
    out["trend"]     = ts["trend"]
    out["strength"]  = ts["strength"]
    out["position"]  = np.where(agree, pos, 0.0)
    out["vetoed"]    = pos.notna() & ~agree
    out.attrs.update(sig.attrs)
    out.attrs.update(ts.attrs)          # method/lookback/t_min, for describe_gate
    out.attrs.update(gated=True, label=label or sig.attrs.get("label", ""))
    return out


def describe_gate(gated, label=None):
    """What the filter actually removed. Returns the frame unchanged."""
    label = label or gated.attrs.get("label", "")
    n     = int(gated["coin"].notna().sum())
    kept  = int((gated["position"] != 0).sum())
    tr    = gated["trend"]
    print(f"{label}  regime filter ({gated.attrs.get('method')}, "
          f"lookback={gated.attrs.get('lookback_days')} DAYS, "
          f"t_min={gated.attrs.get('t_min')}, "
          f"{gated.attrs.get('n_days', '?')} daily closes)")
    print(f"  bars w/ flip : {n:,}")
    print(f"  regime       : bull {int((tr == BULL).sum()):,} | "
          f"bear {int((tr == BEAR).sum()):,} | neither {int((tr == FLAT).sum()):,}")
    print(f"  kept         : {kept:,} ({100 * kept / max(n, 1):.1f}% of flips)")
    print(f"  vetoed       : {int(gated['vetoed'].sum()):,}")
    if kept:
        longs = int((gated["position"] == 1).sum())
        print(f"  direction    : {longs:,} long / {kept - longs:,} short   "
              f"(every one points WITH the trend - the coin chose none of this)")
    return gated


def compare_gate(bars, symbol, seed=FLIP_SEED, account=None, **kw):
    """Three arms on the same bars. Returns a one-row-per-arm table.

        raw        every flip traded            - the null this system starts from
        gated      flip kept only when it agrees with the regime
        trend      every regime signal traded, coin ignored  <- the real benchmark

    If `gated` and `trend` earn the same per trade, the coin is doing nothing
    except throwing away half the signals, and the filter is the whole strategy.
    """
    tf  = bars.attrs.get("tf")
    sig = coin_flip_signals(bars, symbol, tf=tf, seed=seed)
    ts  = trend_state(bars, symbol, **kw)

    arms = {"raw": sig, "gated": gate_signals(sig, ts, symbol)}
    trend_only = sig.copy()
    trend_only["position"] = ts["trend"].astype("float64").replace(0.0, 0.0)
    arms["trend"] = trend_only

    rows = []
    for name, s in arms.items():
        tr, sk, bk = run_session_trades(bars, s, symbol, account=account)
        if len(tr) == 0:
            rows.append(dict(arm=name, trades=0, note=(sk["reason"].mode()[0]
                                                       if len(sk) else "none")))
            continue
        net = tr["net"]
        rows.append(dict(
            arm=name, trades=len(tr), net=net.sum(), per_trade=net.mean(),
            win_pct=100 * (net > 0).mean(),
            max_dd=(tr["equity"].cummax() - tr["equity"]).max(),
            t_stat=net.mean() / (net.std() / np.sqrt(len(net))) if len(net) > 1 else np.nan,
            killed=bk.killed, note=""))
    return pd.DataFrame(rows).set_index("arm")


def sweep_lookback(bars, symbol, days=(5, 10, 20, 40, 80, 160), seed=FLIP_SEED,
                   account=None, **kw):
    """Net per trade against lookback. Parameter sensitivity, in one table.

    A regime that only pays at one setting is a fit to this sample, not a
    regime. What you want to see is a broad plateau with the same sign; what
    you do NOT want is one spike surrounded by losses. `trend` is shown beside
    `gated` at every setting, because if they track each other the coin is
    still contributing nothing no matter how the lookback is tuned.
    """
    rows = []
    for n in days:
        c = compare_gate(bars, symbol, seed=seed, account=account,
                         lookback_days=n, **kw)
        row = {"lookback_days": n}
        for arm in ("raw", "gated", "trend"):
            if arm in c.index and c.loc[arm, "trades"]:
                row[f"{arm}_trades"] = int(c.loc[arm, "trades"])
                row[f"{arm}_per_trade"] = c.loc[arm, "per_trade"]
            else:
                row[f"{arm}_trades"], row[f"{arm}_per_trade"] = 0, np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("lookback_days")


def check_trend_filter(days=600, symbol="/NQ", seed=FLIP_SEED):
    """Falsifiable self-test. One line per property, raises if any is broken."""
    tz  = SESSION["tz"]
    idx = pd.date_range("2022-01-03 00:00", periods=days * 24, freq="60min", tz=tz)
    idx = idx[idx.dayofweek < 5]
    rng = np.random.default_rng(23)
    # a deliberate up-leg then a down-leg, so both regimes actually occur
    drift = np.concatenate([np.full(len(idx) // 2, 6e-5),
                            np.full(len(idx) - len(idx) // 2, -6e-5)])
    px    = 18_000 * np.exp(np.cumsum(rng.normal(0, 9e-4, len(idx)) + drift))
    rad   = np.abs(rng.normal(0, 10, len(idx))) + 3
    bars  = pd.DataFrame({"open": px, "high": px + rad, "low": px - rad,
                          "close": px + rng.normal(0, 5, len(idx)),
                          "volume": rng.integers(1, 9999, len(idx))}, index=idx)
    bars["high"] = bars[["open", "high", "close"]].max(axis=1)
    bars["low"]  = bars[["open", "low",  "close"]].min(axis=1)
    bars.attrs["tf"] = "60m"

    sig    = coin_flip_signals(bars, symbol, tf="60m", seed=seed)
    ts     = trend_state(bars, symbol)
    gated  = gate_signals(sig, ts, symbol)
    et     = bars.index.tz_convert(tz)
    checks = []

    # 1. the gate only subtracts - it must never hand back the opposite side
    kept = gated["position"] != 0
    checks.append(("gate never reverses a flip: kept position == the coin's",
                   bool((gated.loc[kept, "position"] == gated.loc[kept, "coin"]).all())))

    # 2. the stated truth table, all four corners plus neutral
    tr, co, po = gated["trend"], gated["coin"], gated["position"]
    rules = [((tr == BULL) & (co == 1),  1), ((tr == BULL) & (co == -1), 0),
             ((tr == BEAR) & (co == -1), -1), ((tr == BEAR) & (co == 1),  0),
             ((tr == FLAT) & co.notna(),  0)]
    checks.append(("bull+heads=long, bull+tails=flat, bear+tails=short, "
                   "bear+heads=flat, neither=flat",
                   all(bool((po[m] == want).all()) for m, want in rules if m.any())))

    # 3. every surviving trade points WITH the trend - the point of the filter
    checks.append(("every kept trade agrees with the regime",
                   bool((gated.loc[kept, "position"] == gated.loc[kept, "trend"]).all())))

    # 4. it is a DAILY regime: one value per session, constant inside it
    per_day = ts["trend"].groupby(et.normalize()).nunique()
    checks.append((f"regime is constant within a session ({len(per_day):,} days)",
                   bool((per_day <= 1).all())))

    # 5. NO LOOK-AHEAD: rewriting the future must not move a past regime
    cut    = int(len(bars) * 0.6)
    future = bars.copy()
    future.iloc[cut:, :4] *= rng.uniform(0.5, 1.5, size=(len(bars) - cut, 4))
    ts2    = trend_state(future, symbol)
    checks.append(("regime before a cut is unchanged when the future is rewritten",
                   bool((ts2["trend"].iloc[:cut].to_numpy()
                         == ts["trend"].iloc[:cut].to_numpy()).all())))

    # 6. a session's regime may not contain ANY of that session's own bars -
    #    the entry happens before the session opens
    d_all  = pd.DatetimeIndex(et.normalize())
    target = d_all[cut]
    poke   = bars.copy()
    onday  = np.asarray(d_all == target)      # DatetimeIndex == scalar is already ndarray
    poke.iloc[onday, :4] *= 1.30                       # rewrite the WHOLE session
    ts3    = trend_state(poke, symbol)
    checks.append(("a session's regime ignores every bar inside that session",
                   bool((ts3["trend"].to_numpy()[onday]
                         == ts["trend"].to_numpy()[onday]).all())))

    # 7. the daily t-stat matches a reference fit on the same daily closes
    n     = TREND["lookback_days"]
    daily = session_closes(bars)
    y     = np.log(daily.to_numpy())
    ref   = []
    for p in (n + 3, len(y) // 2, len(y) - 1):
        w    = y[p - n + 1:p + 1]
        x    = np.arange(n, dtype=float)
        b, a = np.polyfit(x, w, 1)
        res  = w - (a + b * x)
        se   = np.sqrt((res @ res) / (n - 2) / (((x - x.mean()) ** 2).sum()))
        ref.append(abs(_rolling_slope_t(pd.Series(y, index=daily.index), n)[0].iloc[p]
                       - b / se))
    checks.append((f"daily t-stat matches np.polyfit (max err {max(ref):.2e})",
                   max(ref) < 1e-6))

    # 8. the filter must actually bite
    frac = float(kept.sum()) / float(gated["coin"].notna().sum())
    checks.append((f"filter removes a real share of flips (kept {100*frac:.0f}%)",
                   0.05 < frac < 0.60))

    # 9. a flat/neutral regime trades nothing at all
    allflat = ts.copy(); allflat["trend"] = np.int8(FLAT)
    checks.append(("a market with no regime produces no trades",
                   int((gate_signals(sig, allflat, symbol)["position"] != 0).sum()) == 0))

    # 10. a longer lookback must look back further - warm-up scales with it
    warm = {n: int((trend_state(bars, symbol, lookback_days=n)["trend"] == FLAT)
                   .to_numpy()[:len(bars) // 2].sum()) for n in (10, 160)}
    checks.append((f"a longer lookback warms up later ({warm[10]:,} vs {warm[160]:,} flat bars)",
                   warm[160] > warm[10]))

    print(f"trend filter self-test  ({len(bars):,} synthetic 60m bars over "
          f"{len(daily):,} days, {TREND['method']}, "
          f"lookback={TREND['lookback_days']} days)")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise AssertionError(f"trend filter self-test FAILED: {failed}")
    print("  multi-day regime holds: subtractive, session-constant, blind to its own day")
    describe_gate(gated, symbol)
    return gated


_ = check_trend_filter()

# Usage, once Module.2/3 have the bars and the strategy:
#   ts    = trend_state(nq_1h, "/NQ")
#   sig   = coin_flip_signals(nq_1h, "/NQ", tf="1h")
#   gated = describe_gate(gate_signals(sig, ts, "/NQ"))
#   tr, sk, bk = run_session_trades(nq_1h, gated, "/NQ")
#
#   compare_gate(nq_1h, "/NQ")      # raw vs gated vs trend-only - read this one


In [ ]:
# cellblock.2

# feature engineering (2)  -  a pre-open EMA gate
#     ema_state()  : is the pre-open EMA above or below YESTERDAY's EMA close
#
#   EMA(pre-open) > EMA(yesterday's close)  -> bull -> heads trades LONG
#   EMA(pre-open) < EMA(yesterday's close)  -> bear -> tails trades SHORT
#   inside the deadband                     -> flat -> no trade either way
#
# It plugs into the same gate_signals() as cellblock.1, so the truth table,
# the subtractive property and the downstream engine are unchanged. Only the
# definition of "regime" differs, and that difference is the point of having
# two features: they use DIFFERENT INFORMATION.
#
#   cellblock.1 slope_t : daily closes only, fixed for the whole session. It
#                         knows nothing that happened after yesterday's close.
#   this EMA gate       : the reference is yesterday's EMA close, but the live
#                         side is the EMA as of the last bar BEFORE the entry -
#                         so it does see this morning's pre-market tape.
#
# That extra information is legitimate (those bars have closed) and it is also
# exactly where look-ahead would hide, so ema_state() shifts one bar and the
# self-test proves it: rewrite the entry bar's own close and the state must not
# move. If it ever does, the gate is reading a price it could not have had.
#
# What it cannot do is manufacture edge. Like any gate it only subtracts, so
# every surviving trade points the way the EMA already pointed, and the honest
# benchmark stays "EMA direction traded every day, coin ignored".

EMA = {
    "span":          20,     # bars in the EMA, on the bar timeframe being traded
    "deadband":  0.0005,     # |ema/ref - 1| below this is FLAT (5 bp). 0 = pure sign
    "day_cutoff": "16:00",   # what counts as "yesterday's close", ET
}


def ema_state(bars, label="", span=None, deadband=None, day_cutoff=None):
    """Pre-open EMA vs yesterday's EMA close. Returns a frame on bars.index.

        trend       +1 bull / -1 bear / 0 flat   - usable AT that bar's open
        strength    ema / ref - 1, the fractional gap
        ema         the EMA as of the PREVIOUS bar (what the open can know)
        ref         yesterday's EMA at the cash close

    Two different shifts, for two different reasons, and both matter:

      ema.shift(1)   the EMA through bar t contains bar t's close, which has not
                     happened when bar t OPENS - and the open is when
                     cellblock.2 enters. Shifting makes the live side the EMA
                     through the last COMPLETED bar, i.e. the pre-market tape.

      ref.shift(1)   grouped by session, so a bar on day D references day D-1's
                     EMA at 16:00 ET. Not D's own close (that is the future) and
                     not D-1's last overnight print (that is not "the close").
    """
    span     = int(span if span is not None else EMA["span"])
    deadband = float(deadband if deadband is not None else EMA["deadband"])
    cutoff   = day_cutoff or EMA["day_cutoff"]
    if not isinstance(bars.index, pd.DatetimeIndex):
        raise TypeError(f"{label}: index must be a DatetimeIndex")
    if bars.index.tz is None:
        raise ValueError(f"{label}: bar index is tz-naive - 'yesterday's close' has "
                         "no meaning without a zone. Localize it first.")
    if span < 2:
        raise ValueError("span must be >= 2")

    tz     = SESSION["tz"]
    close  = bars["close"].astype("float64")
    ema    = close.ewm(span=span, adjust=False).mean()

    et     = bars.index.tz_convert(tz)
    hh, mm = (int(x) for x in str(cutoff).split(":"))
    keep   = (et.hour * 60 + et.minute) <= (hh * 60 + mm)
    if not keep.any():
        raise ValueError(f"{label}: no bars at or before {cutoff} {tz}")
    day = et.normalize()

    ema_at_close = ema[keep].groupby(day[keep]).last()      # today's EMA at 16:00
    ref_by_day   = ema_at_close.shift(1)                    # -> YESTERDAY's
    ref          = pd.Series(ref_by_day.reindex(day).to_numpy(), index=bars.index)
    live         = ema.shift(1)                             # pre-open EMA

    gap   = live / ref - 1.0
    state = pd.Series(FLAT, index=bars.index, dtype="int8")
    state[gap > deadband]  = BULL
    state[gap < -deadband] = BEAR
    state[gap.isna()]      = FLAT

    out = pd.DataFrame(index=bars.index.copy())
    out["trend"]    = state
    out["strength"] = gap
    out["ema"]      = live
    out["ref"]      = ref
    out.index.name  = "ts"
    out.attrs.update(label=label, method="ema_preopen", span=span,
                     deadband=deadband, day_cutoff=cutoff,
                     lookback_days=None, t_min=None, n_days=int(len(ema_at_close)))
    return out


def describe_ema(st, label=None):
    """What the EMA gate sees. Returns the frame unchanged."""
    label = label or st.attrs.get("label", "")
    tr, g = st["trend"], st["strength"].dropna()
    print(f"{label}  pre-open EMA gate (span={st.attrs['span']}, "
          f"deadband={st.attrs['deadband']:.4%}, {st.attrs['n_days']:,} sessions)")
    print(f"  bars        : {len(st):,}")
    print(f"  regime      : bull {int((tr == BULL).sum()):,} | "
          f"bear {int((tr == BEAR).sum()):,} | flat {int((tr == FLAT).sum()):,}")
    if len(g):
        print(f"  ema vs ref  : median {g.median():+.4%}, "
              f"p5 {g.quantile(.05):+.4%}, p95 {g.quantile(.95):+.4%}")
    return st


def check_ema_filter(days=400, symbol="/NQ", seed=FLIP_SEED):
    """Falsifiable self-test. One line per property, raises if any is broken."""
    tz  = SESSION["tz"]
    idx = pd.date_range("2023-01-03 00:00", periods=days * 24, freq="60min", tz=tz)
    idx = idx[idx.dayofweek < 5]
    rng = np.random.default_rng(31)
    drift = np.concatenate([np.full(len(idx) // 2, 5e-5),
                            np.full(len(idx) - len(idx) // 2, -5e-5)])
    px    = 18_000 * np.exp(np.cumsum(rng.normal(0, 9e-4, len(idx)) + drift))
    rad   = np.abs(rng.normal(0, 10, len(idx))) + 3
    bars  = pd.DataFrame({"open": px, "high": px + rad, "low": px - rad,
                          "close": px + rng.normal(0, 5, len(idx)),
                          "volume": rng.integers(1, 9999, len(idx))}, index=idx)
    bars["high"] = bars[["open", "high", "close"]].max(axis=1)
    bars["low"]  = bars[["open", "low",  "close"]].min(axis=1)
    bars.attrs["tf"] = "60m"

    sig    = coin_flip_signals(bars, symbol, tf="60m", seed=seed)
    st     = ema_state(bars, symbol)
    gated  = gate_signals(sig, st, symbol)
    et     = bars.index.tz_convert(tz)
    checks = []

    # 1. the live EMA is the one through the PREVIOUS bar, never this one
    ref_ema = bars["close"].astype("float64").ewm(span=EMA["span"], adjust=False).mean()
    checks.append(("live EMA == pandas ewm shifted one bar",
                   bool(np.allclose(st["ema"].to_numpy()[1:],
                                    ref_ema.to_numpy()[:-1], equal_nan=True))))

    # 2. the reference really is YESTERDAY's EMA at the cutoff, not today's
    d_all = pd.DatetimeIndex(et.normalize())
    uniq  = d_all.unique()
    d1, d0 = uniq[80], uniq[79]
    prev_close_ema = ref_ema[(d_all == d0) & ((et.hour * 60 + et.minute) <= 16 * 60)].iloc[-1]
    checks.append(("reference == previous session's EMA at 16:00",
                   bool(np.isclose(st.loc[d_all == d1, "ref"].iloc[0], prev_close_ema))))

    # 3. NO LOOK-AHEAD: rewriting the future must not move a past state
    cut    = int(len(bars) * 0.6)
    future = bars.copy()
    future.iloc[cut:, :4] *= rng.uniform(0.5, 1.5, size=(len(bars) - cut, 4))
    st2    = ema_state(future, symbol)
    checks.append(("state before a cut is unchanged when the future is rewritten",
                   bool((st2["trend"].iloc[:cut].to_numpy()
                         == st["trend"].iloc[:cut].to_numpy()).all())))

    # 4. the state AT a bar may not contain that bar's own close
    poke = bars.copy()
    k    = cut + 7
    poke.iloc[k, poke.columns.get_loc("close")] *= 1.20
    st3  = ema_state(poke, symbol)
    checks.append(("state AT a bar ignores that bar's own close",
                   bool(st3["trend"].iloc[k] == st["trend"].iloc[k])))

    # 5. the stated rule, all four corners
    tr, co, po = gated["trend"], gated["coin"], gated["position"]
    rules = [((tr == BULL) & (co == 1),  1), ((tr == BULL) & (co == -1), 0),
             ((tr == BEAR) & (co == -1), -1), ((tr == BEAR) & (co == 1),  0)]
    checks.append(("EMA up + heads = long, EMA up + tails = flat, "
                   "EMA down + tails = short, EMA down + heads = flat",
                   all(bool((po[m] == want).all()) for m, want in rules if m.any())))

    # 6. subtractive only - a kept trade is always the coin's own side
    kept = po != 0
    checks.append(("gate never reverses a flip",
                   bool((po[kept] == co[kept]).all())))

    # 7. a deadband wide enough to swallow every move trades nothing
    checks.append(("an enormous deadband produces no trades",
                   int((gate_signals(sig, ema_state(bars, symbol, deadband=10.0),
                                     symbol)["position"] != 0).sum()) == 0))

    # 8. the gate must actually bite
    frac = float(kept.sum()) / float(co.notna().sum())
    checks.append((f"filter removes a real share of flips (kept {100*frac:.0f}%)",
                   0.05 < frac < 0.60))

    print(f"pre-open EMA gate self-test  ({len(bars):,} synthetic 60m bars, "
          f"span={EMA['span']}, deadband={EMA['deadband']:.2%})")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise AssertionError(f"EMA gate self-test FAILED: {failed}")
    print("  the EMA gate subtracts only, and cannot see the bar it acts on")
    describe_ema(st, symbol)
    return gated


_ = check_ema_filter()

# Usage:
#   st    = describe_ema(ema_state(nq_1h, "/NQ"))
#   gated = gate_signals(coin_flip_signals(nq_1h, "/NQ", tf="1h"), st, "/NQ")
#   run_session_trades(nq_1h, gated, "/NQ")
# cellblock.3 runs it against the slope_t feature and the raw flip.


In [ ]:
# cellblock.3

# P&L results  -  both features, the raw flip, and the benchmarks that judge them
#     pnl_results()    : one row per (dataset, arm) - the numbers
#     plot_arms_grid() : one panel per dataset, the three CANDIDATES overlaid
#
# Five arms. Three are candidates you might trade, two exist only to judge them:
#     raw        every flip traded          - the null the whole system starts from
#     ts_gate    flip kept when it agrees with the multi-day slope_t regime
#     ema_gate   flip kept when it agrees with the pre-open EMA
#     ts_only    slope_t direction traded every day, coin ignored     [benchmark]
#     ema_only   EMA direction traded every day, coin ignored         [benchmark]
#
# Read it in this order, and the order matters:
#   1. does either gate beat RAW?            - did filtering remove bad trades
#   2. does a gate beat ITS OWN benchmark?   - does the coin still earn its place
# A gate that beats raw but matches its own *_only arm has not found anything;
# it has found the feature, and the coin is just discarding half the signals.
#
# Only the three candidates are drawn. A fourth and fifth line would need
# categorical slots 4-5, and slot 4 puts yellow beside orange, which fails the
# all-pairs colour-blind floor - so the benchmarks live in the table instead of
# being made unreadable on the chart. Slots 1-3 validate all-pairs in both modes
# (worst CVD dE 9.2 light / 9.4 dark, normal-vision 24.0 / 20.9). Aqua sits at
# 2.74:1 on the light surface, under the 3:1 bar, so the relief rule applies:
# every panel carries its values as visible text and every figure has the table
# behind it - identity is never left to colour alone.

ARMS_CHART = ("raw", "ts_gate", "ema_gate")          # candidates - drawn
ARMS_BENCH = ("ts_only", "ema_only")                 # judges - table only
ARMS       = ARMS_CHART + ARMS_BENCH
PAIRED     = {"ts_gate": "ts_only", "ema_gate": "ema_only"}
ARM_COLOR = {
    "light": {"raw": "#2a78d6", "ts_gate": "#eb6834", "ema_gate": "#1baf7a"},
    "dark":  {"raw": "#3987e5", "ts_gate": "#d95926", "ema_gate": "#199e70"},
}


def _arm_signals(bars, symbol, seed=FLIP_SEED, ts_kw=None, ema_kw=None):
    """The five signal frames every result in this cellblock is built from."""
    sig = coin_flip_signals(bars, symbol, tf=bars.attrs.get("tf"), seed=seed)
    ts  = trend_state(bars, symbol, **(ts_kw or {}))
    em  = ema_state(bars, symbol, **(ema_kw or {}))

    def _only(state):
        """Trade the feature's direction every day, coin ignored."""
        f = sig.copy()
        f["position"] = state["trend"].astype("float64")
        return f

    return {"raw":      sig,
            "ts_gate":  gate_signals(sig, ts, symbol),
            "ema_gate": gate_signals(sig, em, symbol),
            "ts_only":  _only(ts),
            "ema_only": _only(em)}


def pnl_results(sets=None, seed=FLIP_SEED, account=None, **kw):
    """Run all three arms over every dataset. Returns a (dataset, arm) table.

    A dataset that cannot produce trades still gets rows, with the reason in
    `note` - an asset missing from a results table reads as "no result" when
    the truth is usually "this data cannot answer the question".
    """
    sets = collect_assets(quiet=True) if sets is None else sets
    rows = []
    for lab, (sym, bars) in sets.items():
        if not INSTRUMENTS[sym]["tradable"]:
            rows.append(dict(dataset=lab, arm="-", trades=0,
                             note=f"{INSTRUMENTS[sym]['unit']} - cannot hold a unit"))
            continue
        try:
            arms = _arm_signals(bars, sym, seed=seed, **kw)
        except Exception as exc:
            rows.append(dict(dataset=lab, arm="-", trades=0,
                             note=f"{type(exc).__name__}: {str(exc).splitlines()[0][:60]}"))
            continue
        for arm in ARMS:
            tr, sk, bk = run_session_trades(bars, arms[arm], sym, account=account)
            if len(tr) == 0:
                rows.append(dict(dataset=lab, arm=arm, trades=0,
                                 note=(sk["reason"].mode()[0] if len(sk) else "no sessions")))
                continue
            net = tr["net"]
            rows.append(dict(
                dataset=lab, arm=arm, trades=len(tr), net=net.sum(),
                per_trade=net.mean(), win_pct=100 * (net > 0).mean(),
                max_dd=(tr["equity"].cummax() - tr["equity"]).max(),
                t_stat=net.mean() / (net.std() / np.sqrt(len(net))) if len(net) > 1 else np.nan,
                killed=bk.killed, note=""))
    out = pd.DataFrame(rows)
    return out.set_index(["dataset", "arm"]) if len(out) else out


def plot_arms_grid(sets=None, seed=FLIP_SEED, account=None, theme="light",
                   ncols=3, panel=(4.8, 3.0), **kw):
    """One panel per dataset, the three arms overlaid. Returns the same table.

    Faceted by dataset rather than drawn on one axis: the assets do not share a
    scale or a period, so one chart of thirty-six curves would say nothing. The
    three arms DO share a scale inside a panel, which is the comparison that
    matters, so they are overlaid there.
    """
    a    = {**ACCOUNT, **(account or {})}
    t    = VIZ[theme]
    col  = ARM_COLOR[theme]
    sets = collect_assets(quiet=True) if sets is None else sets
    if not sets:
        print("no datasets found - run Module.2 first")
        return pd.DataFrame()

    nrows = math.ceil(len(sets) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel[0] * ncols, panel[1] * nrows),
                             facecolor=t["surface"], squeeze=False)
    rows = []
    for ax, (lab, (sym, bars)) in zip(axes.ravel(), sets.items()):
        if not INSTRUMENTS[sym]["tradable"]:
            _blank_panel(ax, t, f"{lab}   no trades",
                         f"{INSTRUMENTS[sym]['unit']}\ncannot hold a unit")
            rows.append(dict(dataset=lab, arm="-", trades=0, net=np.nan,
                             note="cannot hold a unit"))
            continue
        try:
            arms = _arm_signals(bars, sym, seed=seed, **kw)
        except Exception as exc:
            _blank_panel(ax, t, f"{lab}   no trades",
                         f"{type(exc).__name__}\n{str(exc).splitlines()[0][:60]}")
            rows.append(dict(dataset=lab, arm="-", trades=0, net=np.nan,
                             note=type(exc).__name__))
            continue

        _style(ax, t, small=True)
        ax.axhline(0, color=t["axis"], linewidth=1.0)
        bits, drawn = [], False
        for arm in ARMS:
            tr, sk, bk = run_session_trades(bars, arms[arm], sym, account=account)
            if len(tr) == 0:
                if arm in ARMS_CHART:
                    bits.append(f"{arm} -")
                rows.append(dict(dataset=lab, arm=arm, trades=0, net=np.nan,
                                 note=(sk["reason"].mode()[0] if len(sk) else "none")))
                continue
            x, eq = _equity_path(tr)
            if arm in ARMS_CHART:                     # benchmarks: table, not chart
                ax.plot(x, eq, color=col[arm], linewidth=1.5 if arm != "raw" else 1.2,
                        alpha=1.0 if arm != "raw" else 0.85,
                        zorder=4 if arm != "raw" else 3)
                bits.append(f"{arm} ${eq[-1]:+,.0f}")
                drawn = True
            rows.append(dict(dataset=lab, arm=arm, trades=len(tr), net=eq[-1],
                             per_trade=tr["net"].mean(),
                             max_dd=float((np.maximum.accumulate(eq) - eq).max()),
                             killed=bk.killed, note=""))
        if not drawn:
            _blank_panel(ax, t, f"{lab}   no trades", "no arm produced a trade")
            continue
        # the values in text, not carried by colour - this is the relief rule
        ax.set_title(f"{lab}  {_span(bars)}\n" + "   ".join(bits),
                     color=t["ink"], fontsize=8.5, loc="left", pad=5)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    for ax in axes.ravel()[len(sets):]:
        ax.set_visible(False)

    h = [plt.Line2D([], [], color=col[a_], lw=2.0) for a_ in ARMS_CHART]
    leg = fig.legend(h, ["raw - every flip",
                         "ts_gate - flip agrees with the daily slope regime",
                         "ema_gate - flip agrees with the pre-open EMA"],
                     loc="upper right", frameon=False, fontsize=8.5, ncols=3,
                     bbox_to_anchor=(0.995, 1.0))
    for txt in leg.get_texts():
        txt.set_color(t["secondary"])
    fig.suptitle(f"Cumulative net P&L by arm  -  1 unit each, "
                 f"${risk_per_trade(a):,.0f} risk/trade, "
                 f"{TREND['lookback_days']}-day slope regime vs EMA({EMA['span']}) "
                 f"pre-open, seed {seed}   ·   benchmarks are in the table   ·   "
                 f"panels do NOT share an x axis",
                 color=t["ink"], fontsize=11, x=0.006, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.972))
    plt.show()
    out = pd.DataFrame(rows)
    return out.set_index(["dataset", "arm"]) if len(out) else out


def summarise_arms(res):
    """Collapse the per-dataset table into one verdict per arm. Returns it."""
    live = res[res["trades"] > 0] if "trades" in res else res
    if not len(live):
        print("no arm produced a trade")
        return live
    g = live.groupby("arm").agg(datasets=("trades", "size"), trades=("trades", "sum"),
                                net=("net", "sum"), per_trade=("per_trade", "mean"))
    g = g.reindex([a for a in ARMS if a in g.index])
    print("\nby arm, across datasets.  `net` sums dollars over an $18 ETF and a "
          "$25,000 futures\ncontract, so it means very little - read per_trade, and "
          "even that mixes instruments:")
    print(g.round(2).to_string())

    if "raw" in g.index:
        print("\n  1. does the gate beat the raw flip?")
        for gate in ARMS_CHART:
            if gate == "raw" or gate not in g.index:
                continue
            d = g.loc[gate, "per_trade"] - g.loc["raw", "per_trade"]
            print(f"       {gate:9} {d:+9,.2f} / trade vs raw   "
                  f"{'better' if d > 0 else 'WORSE'}")
    print("\n  2. does the gate beat its OWN benchmark? (if not, the coin is idle)")
    for gate, bench in PAIRED.items():
        if gate in g.index and bench in g.index:
            d = g.loc[gate, "per_trade"] - g.loc[bench, "per_trade"]
            verdict = ("the coin is adding nothing - it only samples the feature"
                       if abs(d) < 5 else "they differ - a seed sweep would say if it is real")
            print(f"       {gate:9} {d:+9,.2f} / trade vs {bench:9}  {verdict}")
    return g


# ---- run it -----------------------------------------------------------------
_sets = collect_assets()
print()
results = plot_arms_grid(_sets)
print(results.round(2).to_string())
_ = summarise_arms(results)

# Deeper cuts, when one panel is worth a closer look:
#   compare_gate(nq_1m, "/NQ", lookback_days=250)
#   sweep_lookback(nq_1m, "/NQ", days=(5, 10, 20, 40, 80, 160, 250))
#   pnl_results(_sets, ema_kw={"span": 50})        # a slower pre-open EMA
#   pnl_results(_sets, ts_kw={"lookback_days": 250})


In [ ]:
# cellblock.4

# head to head  -  all five arms, across many coins, on every dataset
#     arm_sweep()           : (dataset, arm, seed) -> the numbers
#     rank_arms()           : who wins where, and how often
#     head_to_head()        : P(A beats B) over every comparable cell
#     plot_arm_comparison() : one panel per arm, t-stat by dataset
#
# Why this cellblock exists when cellblock.3 already prints a table: three of
# the five arms are SEED-DEPENDENT. raw, ts_gate and ema_gate all depend on
# which coin you happened to flip, and /NQ 1m has already shown a single-seed
# spread from -$45,690 to +$169,830 on identical bars. Ranking arms off one
# draw is reading noise. The two *_only arms ignore the coin entirely, so they
# are deterministic - they are run ONCE and repeated across the seed axis, which
# is also why their spread is zero and should not be read as stability.
#
# The chart ranks by t-statistic, not dollars. per_trade in dollars cannot be
# compared across an $18 ETF and a $25,000 futures contract; a t-stat is
# unit-free and answers the only question worth asking of a backtest arm - is
# this distinguishable from zero at all. |t| < 2 is the noise band, drawn.

SWEEP_SEEDS   = 10          # coins per seed-dependent arm
DETERMINISTIC = ("ts_only", "ema_only")     # coin ignored -> one path, not many
ARM_PATHS     = {}          # (dataset, arm, seed) -> (timestamps, equity)


def arm_sweep(sets=None, seeds=SWEEP_SEEDS, base_seed=FLIP_SEED, account=None,
              ts_kw=None, ema_kw=None, verbose=True, keep_paths=True):
    """Every arm, every dataset, `seeds` coins. Returns a long table.

    The feature states are computed ONCE per dataset and reused across seeds -
    trend_state on 5.3M bars is not something to redo ten times, and the state
    does not depend on the coin anyway.

    keep_paths fills ARM_PATHS with every equity curve as it goes, so
    plot_arm_curves() can draw them without re-running the whole sweep. The
    paths are a few thousand floats each; the alternative is doubling a 50s cell.
    """
    sets = collect_assets(quiet=True) if sets is None else sets
    if keep_paths:
        ARM_PATHS.clear()
    rows = []
    for lab, (sym, bars) in sets.items():
        if not INSTRUMENTS[sym]["tradable"]:
            if verbose:
                print(f"  {lab:12} skipped - {INSTRUMENTS[sym]['unit']}")
            continue
        try:
            ts = trend_state(bars, sym, **(ts_kw or {}))
            em = ema_state(bars, sym, **(ema_kw or {}))
        except Exception as exc:
            if verbose:
                print(f"  {lab:12} skipped - {type(exc).__name__}: "
                      f"{str(exc).splitlines()[0][:50]}")
            continue
        if verbose:
            print(f"  {lab:12} {seeds} coins x 5 arms ...", end="", flush=True)

        for k in range(seeds):
            seed = base_seed + k
            sig  = coin_flip_signals(bars, sym, tf=bars.attrs.get("tf"), seed=seed)
            frames = {"raw": sig,
                      "ts_gate":  gate_signals(sig, ts, sym),
                      "ema_gate": gate_signals(sig, em, sym)}
            if k == 0:                       # deterministic arms: run once
                for nm, st in (("ts_only", ts), ("ema_only", em)):
                    f = sig.copy()
                    f["position"] = st["trend"].astype("float64")
                    frames[nm] = f
            for arm, f in frames.items():
                tr, _sk, bk = run_session_trades(bars, f, sym, account=account)
                if len(tr) == 0:
                    continue
                net = tr["net"]
                if keep_paths:
                    ARM_PATHS[(lab, arm, seed)] = _equity_path(tr)
                rows.append(dict(
                    dataset=lab, symbol=sym, arm=arm, seed=seed, trades=len(tr),
                    net=net.sum(), per_trade=net.mean(),
                    t_stat=net.mean() / (net.std() / np.sqrt(len(net))) if len(net) > 1 else np.nan,
                    max_dd=(tr["equity"].cummax() - tr["equity"]).max(),
                    killed=bool(bk.killed)))
        if verbose:
            print(" done")
    sw = pd.DataFrame(rows)
    # repeat each deterministic arm across the seed axis so every arm is
    # comparable cell-for-cell. The VALUES are identical by construction - that
    # is not stability, it is the absence of a coin.
    if len(sw):
        det = sw[sw["arm"].isin(DETERMINISTIC)]
        reps = []
        for k in range(1, seeds):
            d = det.copy(); d["seed"] = base_seed + k
            reps.append(d)
        if reps:
            sw = pd.concat([sw] + reps, ignore_index=True)
    return sw


def plot_arm_curves(sw, arms=None, theme="light", ncols=3, panel=(4.8, 3.0),
                    paths=None):
    """One panel per dataset, every arm's whole BUNDLE of coin paths overlaid.

    The same chart as cellblock.3, with the single-seed line replaced by all
    `seeds` of them. That is the point: cellblock.3 draws one draw per arm, and
    one draw cannot rank arms whose spread runs tens of thousands of dollars on
    identical bars. Here each arm is a cloud, the median-net coin is drawn bold
    on top, and the question becomes whether one cloud actually sits above
    another rather than whether one lucky line does.

    Deterministic arms (ts_only, ema_only) collapse to a single repeated path -
    visible as one hard line instead of a cloud, which is honest: they have no
    coin to vary, not less risk.

    Only three arms are drawn at a time. A fourth needs categorical slot 4,
    which puts yellow beside orange and fails the all-pairs colour-blind floor;
    pass arms=("raw", "ema_gate", "ema_only") to swap which three you look at.
    """
    paths = ARM_PATHS if paths is None else paths
    if not len(sw) or not paths:
        print("nothing to plot - run arm_sweep(keep_paths=True) first")
        return sw
    arms = tuple(arms or ARMS_CHART)[:3]
    t    = VIZ[theme]
    col  = {a: c for a, c in zip(arms, [ARM_COLOR[theme][k] for k in ARMS_CHART])}

    order = [d for d in sw["dataset"].unique()]
    nrows = math.ceil(len(order) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel[0] * ncols, panel[1] * nrows),
                             facecolor=t["surface"], squeeze=False)
    for ax, ds in zip(axes.ravel(), order):
        _style(ax, t, small=True)
        ax.axhline(0, color=t["axis"], linewidth=1.0)
        bits = []
        for arm in arms:
            cells = sw[(sw["dataset"] == ds) & (sw["arm"] == arm)]
            if not len(cells):
                bits.append(f"{arm} -")
                continue
            seeds_here = sorted(cells["seed"].unique())
            for sd in seeds_here:
                pk = paths.get((ds, arm, sd))
                if pk is None:
                    continue
                x, eq = pk
                ax.plot(x, eq, color=col[arm], linewidth=0.7, alpha=0.18, zorder=2)
            # the median-net coin, drawn solid on top
            msd = cells.iloc[(cells["net"] - cells["net"].median()).abs().argsort()
                             ].iloc[0]["seed"]
            pk  = paths.get((ds, arm, msd))
            if pk is not None:
                ax.plot(pk[0], pk[1], color=col[arm], linewidth=1.7, zorder=4)
            bits.append(f"{arm} ${cells['net'].median():+,.0f}")
        ax.set_title(f"{ds}\n" + "   ".join(bits), color=t["ink"],
                     fontsize=8.5, loc="left", pad=5)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    for ax in axes.ravel()[len(order):]:
        ax.set_visible(False)

    h = [plt.Line2D([], [], color=col[a], lw=2.0) for a in arms]
    leg = fig.legend(h, [f"{a}  ({'one path - no coin' if a in DETERMINISTIC else 'median bold'})"
                         for a in arms],
                     loc="upper right", frameon=False, fontsize=8.5, ncols=3,
                     bbox_to_anchor=(0.995, 1.0))
    for txt in leg.get_texts():
        txt.set_color(t["secondary"])
    fig.suptitle(f"Cumulative net P&L, every coin drawn  -  "
                 f"{sw['seed'].nunique()} coins per arm, 1 unit each, "
                 f"${risk_per_trade():,.0f} risk/trade.  Medians are in the titles; "
                 f"panels do NOT share an x axis.",
                 color=t["ink"], fontsize=11, x=0.006, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.972))
    plt.show()
    return sw


def rank_arms(sw):
    """Median per arm per dataset, plus how often each arm wins. Returns the table."""
    if not len(sw):
        print("nothing to rank"); return sw
    med = (sw.groupby(["dataset", "arm"])
             .agg(trades=("trades", "median"), per_trade=("per_trade", "median"),
                  t_stat=("t_stat", "median"), max_dd=("max_dd", "median"),
                  killed=("killed", "mean"))
             .reset_index())
    print("\nmedian across coins, by dataset and arm:")
    piv = med.pivot(index="dataset", columns="arm", values="per_trade")
    piv = piv.reindex(columns=[a for a in ARMS if a in piv.columns])
    print(piv.round(2).to_string())

    best = piv.idxmax(axis=1)
    print("\nbest arm per dataset (median $/trade):")
    for ds, arm in best.items():
        print(f"  {ds:10} {arm:9} {piv.loc[ds, arm]:+9,.2f}")
    print("\nwins:", ", ".join(f"{a} {int((best == a).sum())}"
                               for a in ARMS if (best == a).any()))

    print("\noverall, median over every (dataset, coin) cell:")
    tot = (sw.groupby("arm")
             .agg(cells=("per_trade", "size"), per_trade=("per_trade", "median"),
                  t_stat=("t_stat", "median"), killed=("killed", "mean"))
             .reindex([a for a in ARMS]))
    tot["killed"] = (100 * tot["killed"]).round(0)
    print(tot.rename(columns={"killed": "killed_%"}).round(2).to_string())
    return med


def head_to_head(sw):
    """P(row arm beats column arm) over every shared (dataset, seed). Returns it.

    Paired on the cell, not averaged separately, so the same bars and the same
    coin are on both sides of every comparison.
    """
    if not len(sw):
        print("nothing to compare"); return sw
    wide = sw.pivot_table(index=["dataset", "seed"], columns="arm", values="per_trade")
    arms = [a for a in ARMS if a in wide.columns]
    m = pd.DataFrame(index=arms, columns=arms, dtype="float64")
    for a in arms:
        for b in arms:
            if a == b:
                m.loc[a, b] = np.nan; continue
            both = wide[[a, b]].dropna()
            m.loc[a, b] = 100 * (both[a] > both[b]).mean() if len(both) else np.nan
    print(f"\nP(row beats column), % of {len(wide):,} paired cells:")
    print(m.round(0).to_string())
    return m


def plot_arm_comparison(sw, theme="light", ncols=5, panel=(3.4, 4.2)):
    """One panel per arm: median t-stat by dataset. Returns the plotted frame.

    Faceted by arm with a SHARED x axis - unlike the equity-curve grids, every
    panel here measures the same thing on the same scale, so comparing across
    panels is the point. Bars are sign-coloured (polarity, not identity) and
    every value is printed, so nothing depends on reading a hue.
    """
    if not len(sw):
        print("nothing to plot"); return sw
    t   = VIZ[theme]
    med = (sw.groupby(["dataset", "arm"])["t_stat"].median().unstack("arm")
             .reindex(columns=[a for a in ARMS if a in sw["arm"].unique()]))
    order = list(med.index)[::-1]
    med   = med.loc[order]
    lim   = float(np.nanmax(np.abs(med.to_numpy()))) * 1.35 or 1.0

    fig, axes = plt.subplots(1, len(med.columns),
                             figsize=(panel[0] * len(med.columns), panel[1]),
                             facecolor=t["surface"], sharey=True, squeeze=False)
    for ax, arm in zip(axes.ravel(), med.columns):
        v = med[arm].to_numpy(dtype="float64")
        _style(ax, t, money=False, small=True)
        ax.axvline(0, color=t["axis"], linewidth=1.0)
        for s in (-2, 2):                       # the rough noise band
            ax.axvline(s, color=t["muted"], linewidth=0.9, linestyle=(0, (4, 4)))
        ax.barh(np.arange(len(v)), v, height=0.62,
                color=[t["series"] if (x or 0) >= 0 else t["loss"] for x in v],
                linewidth=0)
        for i, x in enumerate(v):
            if np.isnan(x):
                continue
            ax.annotate(f"{x:+.1f}", (x, i), color=t["ink"], fontsize=7.5,
                        va="center", ha="left" if x >= 0 else "right",
                        xytext=(3 if x >= 0 else -3, 0), textcoords="offset points")
        ax.set_yticks(np.arange(len(order)))
        ax.set_yticklabels(order, fontsize=7.5, color=t["secondary"])
        ax.set_xlim(-lim, lim)
        ax.set_title(arm, color=t["ink"], fontsize=10, loc="left", pad=6)
    fig.suptitle("t-statistic of net P&L per trade, median over coins  -  "
                 "dashed lines are |t| = 2, the noise band.  "
                 "Unit-free, so assets are comparable; nothing here clears it.",
                 color=t["ink"], fontsize=10.5, x=0.006, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()
    return med


# ---- run it -----------------------------------------------------------------
print(f"sweeping {SWEEP_SEEDS} coins x 5 arms over every dataset "
      f"({', '.join(DETERMINISTIC)} are coin-free and run once)")
_sw = arm_sweep(_sets if "_sets" in globals() else None)
_   = rank_arms(_sw)
_   = head_to_head(_sw)
_   = plot_arm_curves(_sw)                       # the bundles, arm by arm
_   = plot_arm_comparison(_sw)                   # the same thing as one t-stat each

# Deeper cuts:
#   arm_sweep(_sets, seeds=50)                       # tighter medians, slower
#   arm_sweep(_sets, ema_kw={"span": 50})            # a slower pre-open EMA
#   arm_sweep(_sets, ts_kw={"lookback_days": 250})   # the 250-day regime
#   plot_arm_curves(_sw, arms=("raw", "ema_gate", "ema_only"))   # swap the three


## Module.5
### bootstrapping

In [ ]:
# cellblock.1

# bootstrapping risk management  -  which capital rule survives a coin flip
#     trade_r()            : realized trades -> R multiples (risk units)
#     kelly_fraction()     : f* = W - (1-W)/R, straight from the sample
#     bootstrap_r()        : block bootstrap over trade SEQUENCES
#     simulate_rule()      : vectorised equity paths under one sizing rule
#     compare_risk_rules() : every rule, ranked by survival
#   (the charts for all of this live in cellblock.2)
#
# ---------------------------------------------------------------------------
# WHAT THIS CAN AND CANNOT ANSWER. Read this before the table.
#
# 1. Risk management cannot change the sign of expectancy. Sizing multiplies
#    every outcome by a positive number, so E[P&L] keeps whatever sign the
#    per-trade edge already had. A fair coin has zero edge before costs and a
#    negative one after them. So the question here is NOT "which rule makes
#    money" - none of them can - but "which rule survives longest, and how
#    deep does it dig". That is what the methodology means by risk management
#    being about making sure losses never become catastrophic.
#
# 2. Resampling realized trades compares CAPITAL ALLOCATION, not STOPS. A
#    different stop rule produces a different set of trades with different R
#    values; you cannot recover it by reshuffling the trades the old stop
#    produced. Comparing ATR stops vs trailing stops vs breakeven stops needs
#    the backtest re-run per rule, which is cellblock.2 work, not bootstrap
#    work. Everything below holds the stop fixed and varies the size.
#
# 3. Once trades are expressed in R, every volatility-normalised sizing scheme
#    collapses onto fixed-fractional. That is what ATR sizing is FOR: it makes
#    each trade risk the same fraction of equity whatever the volatility. So
#    "ATR sizing" does not get its own row - `pct_*` IS it, in R space.
#
# 4. The paths assume a fill at any size. Real size moves the market, and the
#    larger rules below would not fill as modelled. Treat the aggressive rows
#    as upper bounds on how well they could ever do.
# ---------------------------------------------------------------------------

BOOT = {
    "paths":   2_000,     # bootstrap sequences per rule
    "trades":   None,     # length of each path; None = as long as the sample
    "block":      20,     # block bootstrap length. 1 = plain IID resampling
    "seed": 20260918,
}

# Every rule returns DOLLARS AT RISK for the next trade, given where equity is.
# `unit` is one contract's worth of risk under the current stop, so fixed_1u is
# exactly what cellblock.2 of Module.3 does today.
RISK_RULES = {
    "flat":         dict(kind="none"),
    "fixed_1u":     dict(kind="unit"),
    "pct_0.5":      dict(kind="pct", pct=0.005),
    "pct_1":        dict(kind="pct", pct=0.010),
    "pct_2":        dict(kind="pct", pct=0.020),
    "pct_1_whole":  dict(kind="pct", pct=0.010, whole=True),
    "kelly_quarter":dict(kind="kelly", frac=0.25),
    "kelly_half":   dict(kind="kelly", frac=0.50),
    "dd_scaled":    dict(kind="pct", pct=0.010, cut_at=0.05, cut_to=0.5, restore=0.02),
    "pct_1_breaker":dict(kind="pct", pct=0.010, consec=3, cooldown=1),
}


def trade_r(trades):
    """Realized trades -> R multiples: net P&L divided by the risk taken.

    R is the unit the whole methodology is written in, and it is what makes
    different sizing rules comparable: a rule decides how many dollars to put
    at risk, and the trade returns that many dollars times R.

    R is NET of commission and slippage, so a scratch trade is slightly
    negative rather than zero. Losses are NOT clipped at -1: a gap through the
    stop really did cost more than one unit of risk, and those fat left-tail
    observations are the whole reason to bootstrap rather than assume a
    distribution.
    """
    if "planned_risk" not in trades or not len(trades):
        raise ValueError("need a trades frame from run_session_trades()")
    r = (trades["net"] / trades["planned_risk"]).astype("float64")
    return r[np.isfinite(r)].to_numpy()


def kelly_fraction(r):
    """f* = W - (1 - W) / R_ratio, on the sample's own win rate and payoff.

    The methodology's worked example: W=0.55, payoff 1.5 -> f* = 0.25. Same
    formula here, fed by realized trades instead of an assumption.

    A NEGATIVE f* is not an error and not a rounding artifact. It is Kelly
    stating that the bet has negative expectancy and the optimal stake is zero
    - the single most decisive risk-management result available for a strategy
    like this one. simulate_rule() clamps a negative f* to 0, which makes the
    kelly_* rows sit flat by construction. That is the answer, not a bug.
    """
    wins, losses = r[r > 0], r[r <= 0]
    if not len(wins) or not len(losses):
        return dict(W=np.nan, payoff=np.nan, f_star=0.0, expectancy=float(r.mean()))
    W       = len(wins) / len(r)
    payoff  = float(wins.mean() / abs(losses.mean()))
    f_star  = W - (1.0 - W) / payoff if payoff > 0 else -1.0
    return dict(W=float(W), payoff=payoff, f_star=float(f_star),
                expectancy=float(r.mean()))


def bootstrap_r(r, paths=None, trades=None, block=None, seed=None):
    """Block bootstrap: (paths, trades) array of resampled R sequences.

    Blocks rather than IID draws because trade outcomes are not independent -
    they inherit the market's regimes, and a run of losses in one regime is
    exactly the sequence risk that ruins an account. IID resampling breaks
    those runs up and quietly understates drawdown. block=1 gives plain IID
    resampling if you want to see that difference for yourself.
    """
    paths  = int(paths  or BOOT["paths"])
    trades = int(trades or BOOT["trades"] or len(r))
    block  = max(1, int(block or BOOT["block"]))
    rng    = np.random.default_rng(BOOT["seed"] if seed is None else seed)
    n      = len(r)
    if n < 2:
        raise ValueError("need at least 2 trades to bootstrap")

    n_blocks = int(np.ceil(trades / block))
    starts   = rng.integers(0, n, size=(paths, n_blocks))
    offs     = np.arange(block)
    idx      = (starts[:, :, None] + offs[None, None, :]) % n      # wrap around
    return r[idx.reshape(paths, -1)[:, :trades]]


def simulate_rule(seq, rule, account=None, unit_risk=None, f_star=0.0):
    """Equity paths for one sizing rule. Returns a dict of path statistics.

    Vectorised ACROSS paths: one loop over trades, numpy over the 2,000 paths.
    The loop cannot be removed because every rule here is path dependent - size
    depends on equity, equity depends on the previous size.

    Ruin is the same test cellblock.2 of Module.3 uses: drawdown from peak
    equity reaching the account loss limit. It is absorbing - a ruined path
    takes no further trades, which is what the kill switch does.
    """
    a      = {**ACCOUNT, **(account or {})}
    limit  = float(a["loss_limit"])
    unit   = float(unit_risk if unit_risk is not None else risk_per_trade(a))
    cfg    = dict(rule)
    kind   = cfg.get("kind", "pct")
    P, T   = seq.shape

    eq     = np.full(P, limit, dtype="float64")   # start at the loss limit
    peak   = eq.copy()
    alive  = np.ones(P, dtype=bool)
    consec = np.zeros(P, dtype="int32")
    cool   = np.zeros(P, dtype="int32")
    cut    = np.zeros(P, dtype=bool)              # dd_scaled: currently reduced
    ruin_t = np.full(P, -1, dtype="int32")
    taken  = np.zeros(P, dtype="int32")
    curve  = np.empty((P, T + 1), dtype="float64"); curve[:, 0] = eq

    for t in range(T):
        if kind == "none":
            risk = np.zeros(P)
        elif kind == "unit":
            risk = np.full(P, unit)
        elif kind == "kelly":
            risk = max(f_star, 0.0) * float(cfg.get("frac", 1.0)) * eq
        else:                                              # "pct"
            risk = float(cfg["pct"]) * eq
            if "cut_at" in cfg:                            # drawdown scaling
                dd  = (peak - eq) / np.maximum(peak, 1e-9)
                cut = np.where(dd >= cfg["cut_at"], True,
                               np.where(dd <= cfg.get("restore", 0.02), False, cut))
                risk = risk * np.where(cut, cfg.get("cut_to", 0.5), 1.0)
        if cfg.get("whole"):                               # integer contracts
            risk = np.floor(risk / unit) * unit
        if "consec" in cfg:                                # circuit breaker
            risk = np.where(cool > 0, 0.0, risk)

        risk = np.where(alive, np.maximum(risk, 0.0), 0.0)
        pnl  = risk * seq[:, t]
        eq   = eq + pnl
        peak = np.maximum(peak, eq)
        taken += (risk > 0).astype("int32")

        if "consec" in cfg:
            loss   = (pnl < 0) & (risk > 0)
            consec = np.where(loss, consec + 1, np.where(risk > 0, 0, consec))
            cool   = np.where(consec >= cfg["consec"], cfg.get("cooldown", 1),
                              np.maximum(cool - 1, 0))
            consec = np.where(consec >= cfg["consec"], 0, consec)

        newly = alive & ((peak - eq) >= limit)
        ruin_t = np.where(newly, t + 1, ruin_t)
        alive  = alive & ~newly
        eq     = np.where(alive, eq, peak - limit)         # freeze a dead path
        curve[:, t + 1] = eq

    dd = np.maximum.accumulate(curve, axis=1) - curve
    fin = curve[:, -1]
    return dict(final=fin, ret=(fin / limit - 1.0), maxdd=dd.max(axis=1),
                ruined=~alive, ruin_t=ruin_t, taken=taken, curve=curve)


def limit_(a):
    """The account loss limit, for the note strings below."""
    return float(a["loss_limit"])


def compare_risk_rules(trades, rules=None, account=None, paths=None,
                       n_trades=None, block=None, seed=None, label=""):
    """Bootstrap every rule over one trade sample. Returns the ranked table."""
    r  = trade_r(trades)
    k  = kelly_fraction(r)
    a  = {**ACCOUNT, **(account or {})}
    unit = float(trades["planned_risk"].median())

    print(f"{label}  {len(r):,} realized trades, bootstrapped "
          f"{paths or BOOT['paths']:,} x {n_trades or BOOT['trades'] or len(r):,} "
          f"(block={block or BOOT['block']})")
    print(f"  R multiples : mean {r.mean():+.4f}  median {np.median(r):+.4f}  "
          f"sd {r.std():.3f}  min {r.min():+.2f}  max {r.max():+.2f}")
    print(f"  win rate    : {100 * (r > 0).mean():.1f}%   "
          f"payoff {k['payoff']:.3f}   worse than -1R: "
          f"{int((r < -1).sum()):,} trades ({100 * (r < -1).mean():.1f}%)")
    print(f"  Kelly f*    : {k['f_star']:+.4f}"
          + ("   <- NEGATIVE: the optimal stake is ZERO. No sizing rule fixes a "
             "negative edge." if k["f_star"] <= 0 else ""))
    print(f"  expectancy  : {r.mean():+.4f} R/trade = "
          f"${r.mean() * unit:+,.2f} at ${unit:,.0f} risk\n")

    seq  = bootstrap_r(r, paths=paths, trades=n_trades, block=block, seed=seed)
    rows = []
    for name, cfg in (rules or RISK_RULES).items():
        s = simulate_rule(seq, cfg, account=a, unit_risk=unit, f_star=k["f_star"])
        # A rule that never fires sorts to the top on every risk metric, which
        # would read as "best" when it means "did not play". Say which it is.
        note = ""
        if int(np.median(s["taken"])) == 0:
            if cfg.get("kind") == "none":
                note = "baseline - never trades by definition"
            elif cfg.get("kind") == "kelly":
                note = f"NEVER TRADED: Kelly f*={k['f_star']:+.4f} <= 0, stake clamped to 0"
            elif cfg.get("whole"):
                note = (f"NEVER TRADED: {cfg['pct']:.1%} of ${limit_(a):,.0f} = "
                        f"${cfg['pct'] * limit_(a):,.0f} < one ${unit:,.0f} unit")
            else:
                note = "NEVER TRADED"
        rows.append(dict(
            rule=name, note=note,
            ruin_pct=100 * s["ruined"].mean(),
            med_ret=100 * np.median(s["ret"]),
            p5_ret=100 * np.percentile(s["ret"], 5),
            p95_ret=100 * np.percentile(s["ret"], 95),
            cvar5=100 * s["ret"][s["ret"] <= np.percentile(s["ret"], 5)].mean(),
            med_dd=100 * np.median(s["maxdd"]) / a["loss_limit"],
            p95_dd=100 * np.percentile(s["maxdd"], 95) / a["loss_limit"],
            med_trades=int(np.median(s["taken"]))))
    out = pd.DataFrame(rows).set_index("rule")
    out = out.sort_values(["ruin_pct", "med_ret"], ascending=[True, False])
    return out[[c for c in out.columns if c != "note"] + ["note"]]


def check_bootstrap():
    """Falsifiable self-test. One line per property, raises if any is broken."""
    rng, checks = np.random.default_rng(7), []

    # 1. the Kelly formula reproduces the methodology's own worked example
    synth = np.concatenate([np.full(55, 1.5), np.full(45, -1.0)])   # W=.55, payoff 1.5
    checks.append((f"Kelly f* on W=0.55, payoff=1.5 is 0.25 "
                   f"(got {kelly_fraction(synth)['f_star']:.4f})",
                   abs(kelly_fraction(synth)["f_star"] - 0.25) < 1e-9))

    # 2. a losing sample must give a NEGATIVE f* - "do not bet"
    checks.append(("a negative-edge sample yields f* <= 0",
                   kelly_fraction(np.concatenate([np.full(45, 1.0),
                                                  np.full(55, -1.0)]))["f_star"] <= 0))

    # 3. the bootstrap preserves the sample mean (it resamples, it does not shift)
    r  = rng.normal(-0.05, 1.0, 4_000)
    bs = bootstrap_r(r, paths=400, trades=1_500, block=20, seed=1)
    checks.append((f"bootstrap mean tracks the sample "
                   f"({bs.mean():+.4f} vs {r.mean():+.4f})",
                   abs(bs.mean() - r.mean()) < 0.05))

    # 4. every resampled value came from the sample - nothing invented
    checks.append(("every bootstrapped R exists in the original sample",
                   bool(np.isin(bs, r).all())))

    # 5. block=1 is plain IID resampling, and blocks really do sit together
    b1 = bootstrap_r(np.arange(50.0), paths=200, trades=40, block=10, seed=2)
    steps = np.diff(b1.reshape(200, 4, 10), axis=2) % 50
    checks.append(("a block is consecutive in the original order",
                   bool((steps == 1).all())))

    # 6. flat never trades, never moves, never dies
    seq = bootstrap_r(r, paths=200, trades=300, seed=3)
    f   = simulate_rule(seq, RISK_RULES["flat"])
    checks.append(("flat: no trades, no drawdown, no ruin",
                   bool((f["taken"] == 0).all() and (f["maxdd"] == 0).all()
                        and not f["ruined"].any())))

    # 7. constant -1R with fixed units ruins after exactly limit/unit trades
    lose = np.full((5, 200), -1.0)
    unit = risk_per_trade() 
    exp  = int(round(ACCOUNT["loss_limit"] / unit))
    u    = simulate_rule(lose, RISK_RULES["fixed_1u"], unit_risk=unit)
    checks.append((f"fixed_1u dies on trade {exp} when every trade loses 1R "
                   f"(got {int(u['ruin_t'][0])})",
                   bool((u["ruin_t"] == exp).all() and u["ruined"].all())))

    # 8. ruin is absorbing - a dead path takes no further trades
    checks.append(("a ruined path stops trading",
                   bool((u["taken"] <= exp).all())))

    # 9. percentage sizing on a pure loser can never be ruined outright -
    #    it shrinks geometrically. This is the whole point of fractional sizing.
    p = simulate_rule(np.full((5, 400), -1.0), RISK_RULES["pct_1"])
    checks.append(("pct_1 on an all-loss sequence shrinks but survives",
                   bool(not p["ruined"].any() and (p["final"] > 0).all()
                        and (p["final"] < ACCOUNT["loss_limit"]).all())))

    # 10. drawdown scaling really does cut size, so it loses less on the way down
    a_ = simulate_rule(np.full((5, 400), -1.0), RISK_RULES["pct_1"])["final"][0]
    b_ = simulate_rule(np.full((5, 400), -1.0), RISK_RULES["dd_scaled"])["final"][0]
    checks.append((f"dd_scaled loses less than flat pct on a losing run "
                   f"(${b_:,.0f} vs ${a_:,.0f})", b_ > a_))

    # 11. the consecutive-loss breaker actually skips trades
    br = simulate_rule(np.full((5, 200), -1.0), RISK_RULES["pct_1_breaker"])
    nb = simulate_rule(np.full((5, 200), -1.0), RISK_RULES["pct_1"])
    checks.append((f"the breaker takes fewer trades ({int(br['taken'][0])} "
                   f"vs {int(nb['taken'][0])})", br["taken"][0] < nb["taken"][0]))

    # 12. a positive-edge sample must NOT be ruined by fractional sizing
    win = simulate_rule(np.full((5, 300), +1.0), RISK_RULES["pct_1"])
    checks.append(("a winning sequence compounds and never ruins",
                   bool(not win["ruined"].any()
                        and (win["final"] > ACCOUNT["loss_limit"]).all())))

    print("bootstrap risk self-test")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise AssertionError(f"bootstrap self-test FAILED: {failed}")
    print("  sizing, breakers and ruin behave as specified")
    return True


_ = check_bootstrap()

# ---- run it on the longest real sample --------------------------------------
_src = None
for _lab in ("/NQ 1m", "/NQ 1h"):
    _s = (_sets if "_sets" in globals() else collect_assets(quiet=True)).get(_lab)
    if _s is not None:
        _sym, _bars = _s
        _sig = coin_flip_signals(_bars, _sym, tf=_bars.attrs.get("tf"))
        _tr, _sk, _bk = run_session_trades(_bars, _sig, _sym)
        if len(_tr):
            _src = (_lab, _tr)
            break
if _src is None:
    raise RuntimeError("no trade sample - run Module.2/3 first")

print()
risk_table = compare_risk_rules(_src[1], label=f"{_src[0]}  raw coin flip")
print(risk_table.round(2).to_string())

# `_src` is (label, trades) and cellblock.2 charts exactly this sample.
#
# Deeper cuts:
#   compare_risk_rules(_src[1], block=1)        # IID resampling, for the contrast
#   compare_risk_rules(_src[1], n_trades=5000)  # a longer life than the sample had


In [ ]:
# cellblock.2

# bootstrap charts  -  the same rules cellblock.1 tabled, drawn
#     _rule_paths()    : re-runs the bootstrap and keeps only what a chart needs
#     plot_risk_rules(): a fan per rule, then three medians overlaid
#
# Split out from cellblock.1 so the numbers and the pictures can be re-run
# independently: the table is 2s, the figures re-simulate every rule to get the
# paths back and cost a little more. Everything here reads cellblock.1's
# functions and its `_src` sample - nothing is recomputed differently, so the
# ruin and median figures in the titles match the table exactly.
#
# Two reading notes that the table cannot carry:
#   - x is TRADE NUMBER, not time. These are resampled sequences; they have an
#     order but no dates.
#   - a dead path is frozen at the equity it died with and marked with an x.
#     A flat line along the bottom is not an account sitting still, it is an
#     account that has already hit the loss limit and stopped.
_need = ["simulate_rule", "bootstrap_r", "trade_r", "kelly_fraction",
         "RISK_RULES", "BOOT", "_src"]
_missing = [n for n in _need if n not in globals()]
if _missing:
    raise RuntimeError(f"run Module.5 cellblock.1 first - missing {_missing}")

# Overlay colours: categorical slots 1-3, validated all-pairs in both modes
# (worst CVD dE 9.2 light / 9.4 dark, normal-vision 24.0 / 20.9). Only three
# rules are overlaid at a time - a fourth needs slot 4, which puts yellow beside
# orange and fails the colour-blind floor. The facet grid carries all ten.
RULE_COLOR = {"light": ["#2a78d6", "#eb6834", "#1baf7a"],
              "dark":  ["#3987e5", "#d95926", "#199e70"]}


def _rule_paths(trades, rules=None, account=None, paths=None, n_trades=None,
                block=None, seed=None, n_show=150, max_pts=400):
    """Bootstrap once, simulate every rule, keep only what a chart needs.

    The full path array is 2,000 x 2,438 floats PER RULE - about 39 MB each,
    390 MB for ten. So each rule is simulated, reduced to a sampled handful of
    curves plus the percentile envelope, and thrown away before the next one.
    Curves are also thinned along the trade axis: 2,438 points cannot be
    distinguished on a 4-inch panel, and drawing them all just makes it slow.
    """
    a    = {**ACCOUNT, **(account or {})}
    r    = trade_r(trades)
    k    = kelly_fraction(r)
    unit = float(trades["planned_risk"].median())
    seq  = bootstrap_r(r, paths=paths, trades=n_trades, block=block, seed=seed)
    rng  = np.random.default_rng(BOOT["seed"])
    out  = {}
    for name, cfg in (rules or RISK_RULES).items():
        s    = simulate_rule(seq, cfg, account=a, unit_risk=unit, f_star=k["f_star"])
        cur  = s["curve"]
        step = max(1, cur.shape[1] // max_pts)
        pick = rng.choice(cur.shape[0], size=min(n_show, cur.shape[0]), replace=False)
        out[name] = dict(
            x=np.arange(cur.shape[1])[::step],
            sample=cur[pick][:, ::step],
            med=np.median(cur, axis=0)[::step],
            p5=np.percentile(cur, 5, axis=0)[::step],
            p95=np.percentile(cur, 95, axis=0)[::step],
            ruin_pct=100 * s["ruined"].mean(),
            med_ret=100 * np.median(s["ret"]),
            deaths=(s["ruin_t"][s["ruined"]], s["final"][s["ruined"]]),
            note="never traded" if int(np.median(s["taken"])) == 0 else "")
        del s, cur
    out["_meta"] = dict(start=float(a["loss_limit"]), unit=unit, kelly=k,
                        n_paths=seq.shape[0], n_trades=seq.shape[1])
    return out


def plot_risk_rules(trades, rules=None, theme="light", ncols=4, panel=(4.1, 3.3),
                    overlay=("fixed_1u", "pct_1", "pct_0.5"), **kw):
    """Two figures: a fan per rule, then the medians of three rules overlaid.

    The fans answer "how wide is the outcome", which is the only honest way to
    read a bootstrap - a single path says nothing. The overlay answers "which
    rule is better", which needs them on one axis and so is limited to three.

    x is TRADE NUMBER, not time. These are resampled sequences; they have an
    order but no dates. A dead path is frozen at the equity it died with and
    marked, because a flat line at the bottom is not the same as a live account
    sitting still.
    """
    st   = _rule_paths(trades, rules=rules, **kw)
    meta = st.pop("_meta")
    t    = VIZ[theme]
    start, k = meta["start"], meta["kelly"]

    nrows = math.ceil(len(st) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel[0] * ncols, panel[1] * nrows),
                             facecolor=t["surface"], squeeze=False, sharey=True)
    for ax, (name, d) in zip(axes.ravel(), st.items()):
        _style(ax, t, small=True)
        ax.axhline(start, color=t["axis"], linewidth=1.0)
        ax.axhline(0, color=t["loss"], linewidth=1.1, linestyle=(0, (5, 4)))
        for row in d["sample"]:
            ax.plot(d["x"], row, color=t["ensemble"], linewidth=0.6, alpha=0.16, zorder=2)
        ax.fill_between(d["x"], d["p5"], d["p95"], color=t["series"],
                        alpha=0.12, linewidth=0, zorder=3)
        ax.plot(d["x"], d["med"], color=t["series"], linewidth=1.8, zorder=5)
        if len(d["deaths"][0]):
            ax.plot(d["deaths"][0], d["deaths"][1], "x", markersize=3.5,
                    markeredgewidth=1.0, color=t["loss"], alpha=0.5, zorder=6)
        ax.set_title(f"{name}   ruin {d['ruin_pct']:.1f}%   med {d['med_ret']:+.0f}%"
                     + (f"\n{d['note']}" if d["note"] else ""),
                     color=t["loss"] if d["ruin_pct"] > 10 else t["ink"],
                     fontsize=8.5, loc="left", pad=5)
        ax.set_xlabel("trade #", color=t["muted"], fontsize=7.5)
    for ax in axes.ravel()[len(st):]:
        ax.set_visible(False)
    fig.suptitle(f"Equity under each risk rule  -  {meta['n_paths']:,} bootstrap paths "
                 f"x {meta['n_trades']:,} trades, ${start:,.0f} start, "
                 f"${meta['unit']:,.0f} per unit of risk.  Dashed red is zero; "
                 f"x marks a path that hit the loss limit.",
                 color=t["ink"], fontsize=11, x=0.006, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.955))
    plt.show()

    # ---- the head-to-head: medians of three rules on one axis ---------------
    picks = [p for p in overlay if p in st][:3]
    if len(picks) >= 2:
        col = RULE_COLOR[theme]
        fig2, ax = plt.subplots(figsize=(13, 5.5), facecolor=t["surface"])
        _style(ax, t, "Equity ($)")
        ax.axhline(start, color=t["axis"], linewidth=1.0)
        ax.axhline(0, color=t["loss"], linewidth=1.1, linestyle=(0, (5, 4)))
        for c, name in zip(col, picks):
            d = st[name]
            ax.fill_between(d["x"], d["p5"], d["p95"], color=c, alpha=0.10, linewidth=0)
            ax.plot(d["x"], d["med"], color=c, linewidth=2.0,
                    label=f"{name}   ruin {d['ruin_pct']:.1f}%   "
                          f"med {d['med_ret']:+.0f}%")
        # upper right: every curve descends, so that corner is the empty one
        leg = ax.legend(loc="upper right", frameon=False, fontsize=9)
        for txt in leg.get_texts():
            txt.set_color(t["secondary"])
        ax.set_xlabel("trade #", color=t["secondary"], fontsize=9)
        ax.set_title(f"Median equity and the 5-95% band  -  Kelly f* = "
                     f"{k['f_star']:+.4f}, so the optimal stake is "
                     f"{'ZERO and every line below is a loss taken slowly'
                        if k['f_star'] <= 0 else f'{k['f_star']:.1%}'}",
                     color=t["ink"], fontsize=10.5, loc="left", pad=12)
        fig2.tight_layout()
        plt.show()
    return pd.DataFrame({n: {"ruin_pct": d["ruin_pct"], "med_ret": d["med_ret"]}
                         for n, d in st.items()}).T


def check_risk_charts():
    """The reduced curves a chart draws must still be the simulated ones."""
    st   = _rule_paths(_src[1], rules={"pct_1": RISK_RULES["pct_1"]},
                       paths=200, n_trades=300, n_show=20)
    meta = st.pop("_meta")
    d    = st["pct_1"]
    seq  = bootstrap_r(trade_r(_src[1]), paths=200, trades=300)
    ref  = simulate_rule(seq, RISK_RULES["pct_1"],
                         unit_risk=float(_src[1]["planned_risk"].median()),
                         f_star=kelly_fraction(trade_r(_src[1]))["f_star"])
    checks = [
        ("median curve ends where the simulation's median equity ends",
         abs(d["med"][-1] - np.median(ref["curve"][:, -1])) < 1e-6),
        ("the 5-95 band really brackets the median",
         bool((d["p5"] <= d["med"] + 1e-9).all() and (d["p95"] >= d["med"] - 1e-9).all())),
        (f"ruin % in the title matches the table ({d['ruin_pct']:.2f}%)",
         abs(d["ruin_pct"] - 100 * ref["ruined"].mean()) < 1e-9),
        ("thinning kept the first and last trade",
         d["x"][0] == 0 and d["x"][-1] <= ref["curve"].shape[1] - 1),
        ("every drawn path starts at the account balance",
         bool(np.allclose(d["sample"][:, 0], meta["start"]))),
    ]
    print("bootstrap chart self-test")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [n for n, ok in checks if not ok]
    if failed:
        raise AssertionError(f"chart self-test FAILED: {failed}")
    print("  the pictures show the simulation, not a smoothed version of it")
    return True


_ = check_risk_charts()

print()
_ = plot_risk_rules(_src[1])

# Deeper cuts:
#   plot_risk_rules(_src[1], overlay=("pct_2", "pct_1", "dd_scaled"))
#   plot_risk_rules(_src[1], block=1)          # IID resampling, for the contrast
#   plot_risk_rules(_src[1], n_trades=5000)    # a longer life than the sample had


## Module.6
### machine learning filter

In [ ]:
# cellblock.1

# random forest gate  -  can a classifier tell a tradable session from a trap
#     session_features() : causal features + label, one row per session
#     walk_forward_rf()  : expanding-window OOS probabilities, never in-sample
#     rf_state()         : those probabilities as a +1/-1/0 gate
#     best_worst()       : what the best and worst trades had in common
#
#   RF says UP   + heads -> LONG        RF says UP   + tails -> no trade
#   RF says DOWN + tails -> SHORT       RF says DOWN + heads -> no trade
#   RF unsure            -> no trade, whichever way the coin lands
#
# ---------------------------------------------------------------------------
# THE THREE THINGS THAT MAKE OR BREAK THIS. Read before the numbers.
#
# 1. A classifier cannot extract edge from the coin. It predicts the MARKET,
#    not the flip - the flip is independent of everything by construction. So
#    if the forest can call direction, the coin is redundant and you should
#    trade the forest; if it cannot, the gate is noise. Either way the honest
#    benchmark is "forest direction traded every session, coin ignored", and
#    it is reported beside the gated arm every time.
#
# 2. Look-ahead is fatal and easy here. The label is a FORWARD return, so any
#    feature that touches the session it labels leaks the answer. Everything
#    below is built from bars strictly BEFORE the entry bar, training is
#    expanding-window walk-forward with no shuffling, and every number quoted
#    is out-of-sample. check_rf() proves it two ways: poke the labelled
#    session's own bars and the features must not move, and train on SHUFFLED
#    labels - if accuracy stays near 50% the pipeline is clean, and if it
#    does not, something leaks and the result is worthless.
#
# 3. Accuracy is not edge. Session direction is close to a coin already, so
#    the base rate is ~50%. 52% accuracy on a symmetric payoff still loses
#    after costs. The comparison that matters is net P&L per trade against
#    raw, not the confusion matrix.
# ---------------------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RF = {
    "n_estimators":      400,
    "max_depth":           4,   # shallow on purpose: ~2.4k sessions, 12 features
    "min_samples_leaf":   40,   # a leaf must describe a real population
    "max_features":    "sqrt",
    "class_weight": "balanced",
    "n_jobs":             -1,
    "random_state": 20260918,
}
WALK = {
    "min_train":     600,   # sessions before the first prediction is allowed
    "retrain_every": 125,   # refit roughly twice a year of sessions
    "margin":       0.02,   # |p_up - 0.5| must clear this to call a direction
}


def _entry_rows(bars, session=None):
    """Positions of the bars cellblock.2 of Module.3 would actually enter on.

    Same rule, same binary search: the first bar at or after (cash open minus
    entry_lead) and still before the open. Features are then taken from the bar
    BEFORE each of these, so nothing the entry bar itself contains can be used.
    """
    s   = {**SESSION, **(session or {})}
    tz  = s["tz"]
    et  = bars.index.tz_convert(tz)
    out_i, out_day, out_end = [], [], []
    for midnight in et.normalize().unique():
        day      = midnight.date()
        open_ts  = pd.Timestamp(f"{day} {s['cash_open']}",  tz=tz)
        close_ts = pd.Timestamp(f"{day} {s['cash_close']}", tz=tz)
        lo = int(et.searchsorted(open_ts - s["entry_lead"], "left"))
        hi = int(et.searchsorted(open_ts, "left"))
        if hi <= lo:
            continue
        end = max(int(et.searchsorted(close_ts, "right")), lo + 1) - 1
        out_i.append(lo); out_day.append(midnight); out_end.append(end)
    return np.array(out_i), pd.DatetimeIndex(out_day), np.array(out_end)


def session_features(bars, label="", session=None):
    """One row per session: causal features + the forward label. Returns a frame.

    EVERY feature is read at position i-1, the last bar that had closed before
    the entry - which does include this morning's pre-open bars, exactly as the
    EMA gate does, because those bars have finished and are observable when the
    entry is placed. Nothing reads the entry bar itself or anything after it.
    Daily features come from closes through the PREVIOUS session.
    The label is sign(session close - entry open), which is the future - that
    is what makes it a label and why nothing else may touch it.
    """
    if bars.index.tz is None:
        raise ValueError(f"{label}: tz-naive bars have no sessions. Localize first.")
    i, days, end = _entry_rows(bars, session)
    if len(i) < 50:
        raise ValueError(f"{label}: only {len(i)} sessions - not enough to train")

    c    = bars["close"].astype("float64")
    o    = bars["open"].astype("float64").to_numpy()
    atr  = wilder_atr(bars).shift(1).to_numpy()          # context only, shifted
    prev = np.maximum(i - 1, 0)                          # the last CLOSED bar

    daily  = session_closes(bars)                        # from cellblock.1 of Module.4
    dlog   = np.log(daily)
    dret   = dlog.diff()
    hist   = pd.DataFrame({
        "ret_1d":  dret.shift(1),
        "ret_5d":  dlog.diff(5).shift(1),
        "ret_20d": dlog.diff(20).shift(1),
        "rv5":     dret.rolling(5).std().shift(1),
        "rv20":    dret.rolling(20).std().shift(1),
        "hi20":    dlog.rolling(20).max().shift(1),
        "lo20":    dlog.rolling(20).min().shift(1),
        "prev_c":  dlog.shift(1),
    })
    h = hist.reindex(days)

    last_c = c.to_numpy()[prev]
    f = pd.DataFrame(index=bars.index[i])
    f["gap"]       = np.log(last_c) - h["prev_c"].to_numpy()          # overnight move
    f["gap_atr"]   = f["gap"].to_numpy() * last_c / np.where(atr[prev] > 0, atr[prev], np.nan)
    f["atr_pct"]   = atr[prev] / last_c
    f["ret_1d"]    = h["ret_1d"].to_numpy()
    f["ret_5d"]    = h["ret_5d"].to_numpy()
    f["ret_20d"]   = h["ret_20d"].to_numpy()
    f["rv5"]       = h["rv5"].to_numpy()
    f["rv20"]      = h["rv20"].to_numpy()
    f["vol_ratio"] = f["rv5"] / f["rv20"].replace(0, np.nan)
    f["dist_hi20"] = np.log(last_c) - h["hi20"].to_numpy()
    f["dist_lo20"] = np.log(last_c) - h["lo20"].to_numpy()
    f["dow"]       = days.dayofweek.to_numpy()

    fwd        = np.log(c.to_numpy()[end]) - np.log(o[i])   # entry open -> session close
    f["_fwd"]  = fwd                                        # kept for analysis only
    f["_y"]    = (fwd > 0).astype("int8")                   # the label
    f["_entry_pos"] = i
    f.index.name = "ts"
    f.attrs.update(label=label, n_sessions=len(f))
    return f.dropna(subset=[c for c in f.columns if not c.startswith("_")])


FEATURES = ["gap", "gap_atr", "atr_pct", "ret_1d", "ret_5d", "ret_20d",
            "rv5", "rv20", "vol_ratio", "dist_hi20", "dist_lo20", "dow"]


def walk_forward_rf(feat, features=None, min_train=None, retrain_every=None,
                    shuffle_y=False, seed=None, **rf_kw):
    """Expanding-window walk forward. Returns (p_up, folds, importances).

    Fold k trains on rows [0, start) and predicts [start, stop). start only
    ever increases, so no model ever sees a row at or after the ones it scores.
    Nothing is shuffled: shuffling a time series into train/test is the single
    most common way a backtest gets a fake result, because a row from 2024
    would then help predict 2015.

    shuffle_y=True destroys the label/feature relationship while keeping every
    other mechanic identical. Out-of-sample accuracy must collapse to the base
    rate. If it does not, the pipeline leaks and nothing else here is valid.
    """
    cols  = features or FEATURES
    X     = feat[cols]
    y     = feat["_y"].astype(int)
    mt    = int(min_train     or WALK["min_train"])
    re_   = int(retrain_every or WALK["retrain_every"])
    kw    = {**RF, **rf_kw}
    if seed is not None:
        kw["random_state"] = seed
    if shuffle_y:
        y = pd.Series(np.random.default_rng(kw["random_state"]).permutation(y.to_numpy()),
                      index=y.index)
    if len(X) <= mt:
        raise ValueError(f"{len(X)} sessions but min_train={mt} - nothing left to test")

    p, folds, imps = np.full(len(X), np.nan), [], []
    start = mt
    while start < len(X):
        stop = min(start + re_, len(X))
        clf  = RandomForestClassifier(**kw).fit(X.iloc[:start], y.iloc[:start])
        p[start:stop] = clf.predict_proba(X.iloc[start:stop])[:, 1]
        folds.append((0, start, start, stop))
        imps.append(clf.feature_importances_)
        start = stop
    imp = pd.Series(np.mean(imps, axis=0), index=cols).sort_values(ascending=False)
    return pd.Series(p, index=X.index, name="p_up"), folds, imp


def rf_state(bars, label="", margin=None, feat=None, p_up=None, **kw):
    """The forest's call as a gate frame: +1 up / -1 down / 0 unsure.

    Shaped exactly like trend_state() and ema_state() so gate_signals() takes
    it unchanged - the truth table, the subtractive property and the downstream
    engine are all reused rather than reimplemented.

    Sessions before the first fold have no model yet, so they are 0 and simply
    do not trade. That is not a gap in the data, it is the training period.
    """
    m    = float(margin if margin is not None else WALK["margin"])
    feat = session_features(bars, label) if feat is None else feat
    if p_up is None:
        p_up, _folds, _imp = walk_forward_rf(feat, **kw)

    state = pd.Series(FLAT, index=feat.index, dtype="int8")
    state[p_up > 0.5 + m] = BULL
    state[p_up < 0.5 - m] = BEAR
    state[p_up.isna()]    = FLAT

    out = pd.DataFrame(index=bars.index.copy())
    out["trend"]    = state.reindex(bars.index).fillna(FLAT).to_numpy().astype("int8")
    out["strength"] = (p_up - 0.5).reindex(bars.index).to_numpy()
    out.index.name  = "ts"
    out.attrs.update(label=label, method="random_forest", margin=m,
                     lookback_days=None, t_min=None,
                     n_days=int(p_up.notna().sum()))
    return out


def describe_rf(feat, p_up, imp, label=""):
    """Out-of-sample skill, against the base rate. Returns the scored subset."""
    m    = p_up.notna()
    y    = feat.loc[m, "_y"].astype(int)
    pred = (p_up[m] > 0.5).astype(int)
    base = max(y.mean(), 1 - y.mean())
    acc  = float((pred == y).mean())
    auc  = float(roc_auc_score(y, p_up[m])) if y.nunique() > 1 else np.nan
    print(f"{label}  random forest, walk-forward out-of-sample")
    print(f"  sessions   : {len(feat):,} total, {int(m.sum()):,} scored "
          f"({len(feat) - int(m.sum()):,} held back to train the first model)")
    print(f"  label      : {100 * y.mean():.1f}% up sessions  -> base rate {100 * base:.1f}%")
    print(f"  accuracy   : {100 * acc:.1f}%   AUC {auc:.4f}"
          + ("   <- no better than guessing the majority class"
             if acc <= base + 0.005 else "   <- above the base rate"))
    print(f"  calls      : {int((p_up[m] > 0.5 + WALK['margin']).sum()):,} up / "
          f"{int((p_up[m] < 0.5 - WALK['margin']).sum()):,} down / "
          f"{int(((p_up[m] - 0.5).abs() <= WALK['margin']).sum()):,} unsure")
    print(f"  importance : " + ", ".join(f"{k} {v:.3f}" for k, v in imp.head(6).items()))
    return feat.loc[m]


def best_worst(trades, feat, q=0.1):
    """What separated the best trades from the worst. Returns the comparison.

    Deciles of realized net P&L, with the entry conditions each one had. This
    is a DESCRIPTION of the sample, not a rule: pick the conditions that look
    good here and you have fitted the noise. It is useful for one thing - if
    the two columns are the same to within their spread, there was no condition
    to find, and no classifier was ever going to find one.
    """
    j = trades.join(feat[[c for c in FEATURES if c in feat]], how="inner")
    if len(j) < 20:
        print("too few matched trades to compare"); return pd.DataFrame()
    lo, hi = j["net"].quantile(q), j["net"].quantile(1 - q)
    worst, best = j[j["net"] <= lo], j[j["net"] >= hi]
    rows = []
    for c in FEATURES:
        if c not in j:
            continue
        b, w = best[c].astype(float), worst[c].astype(float)
        pooled = j[c].astype(float).std()
        rows.append(dict(feature=c, best=b.median(), worst=w.median(),
                         diff=b.median() - w.median(),
                         diff_sd=(b.median() - w.median()) / pooled if pooled else np.nan))
    out = pd.DataFrame(rows).set_index("feature")
    out = out.reindex(out["diff_sd"].abs().sort_values(ascending=False).index)
    print(f"\nbest vs worst {q:.0%} of trades by net P&L "
          f"({len(best)} vs {len(worst)} trades)")
    print(out.round(4).to_string())
    big = out["diff_sd"].abs().max()
    print(f"  largest separation: {big:.2f} standard deviations"
          + ("   <- nothing here: the conditions are the same on both ends"
             if big < 0.25 else "   <- worth a look, but this is in-sample by "
             "construction"))
    return out


def compare_rf(bars, symbol, seed=FLIP_SEED, account=None, **kw):
    """raw vs rf_gate vs rf_only, on the same bars. Returns the table."""
    feat = session_features(bars, symbol)
    p_up, folds, imp = walk_forward_rf(feat, **kw)
    describe_rf(feat, p_up, imp, symbol)
    st = rf_state(bars, symbol, feat=feat, p_up=p_up)

    sig  = coin_flip_signals(bars, symbol, tf=bars.attrs.get("tf"), seed=seed)
    only = sig.copy(); only["position"] = st["trend"].astype("float64")
    arms = {"raw": sig, "rf_gate": gate_signals(sig, st, symbol), "rf_only": only}

    rows = []
    for name, f in arms.items():
        tr, sk, bk = run_session_trades(bars, f, symbol, account=account)
        if len(tr) == 0:
            rows.append(dict(arm=name, trades=0, note="no trades")); continue
        net = tr["net"]
        rows.append(dict(arm=name, trades=len(tr), net=net.sum(),
                         per_trade=net.mean(), win_pct=100 * (net > 0).mean(),
                         max_dd=(tr["equity"].cummax() - tr["equity"]).max(),
                         t_stat=net.mean() / (net.std() / np.sqrt(len(net))),
                         killed=bk.killed, note=""))
    return pd.DataFrame(rows).set_index("arm"), feat, p_up, imp, st


def check_rf(days=900, symbol="/NQ"):
    """Falsifiable self-test. One line per property, raises if any is broken."""
    tz  = SESSION["tz"]
    idx = pd.date_range("2020-01-02 00:00", periods=days * 24, freq="60min", tz=tz)
    idx = idx[idx.dayofweek < 5]
    rng = np.random.default_rng(41)
    px  = 18_000 * np.exp(np.cumsum(rng.normal(0, 9e-4, len(idx))))
    rad = np.abs(rng.normal(0, 10, len(idx))) + 3
    bars = pd.DataFrame({"open": px, "high": px + rad, "low": px - rad,
                         "close": px + rng.normal(0, 5, len(idx)),
                         "volume": rng.integers(1, 9999, len(idx))}, index=idx)
    bars["high"] = bars[["open", "high", "close"]].max(axis=1)
    bars["low"]  = bars[["open", "low",  "close"]].min(axis=1)
    bars.attrs["tf"] = "60m"

    feat   = session_features(bars, symbol)
    checks = []

    # 1. the label may never appear among the features
    checks.append(("no label column is fed to the model",
                   not any(c.startswith("_") for c in FEATURES)))

    # 2. a session's features may not contain the ENTRY BAR or anything after
    #    it. Bars earlier in the same session are fair game and deliberately
    #    used - they have closed, and the pre-open tape is exactly the
    #    information the EMA gate reads too. So the test rewrites the entry bar
    #    AND the whole future, which is the real causality claim.
    tgt   = feat.index[len(feat) // 2]
    pos   = int(feat.loc[tgt, "_entry_pos"])
    poke  = bars.copy()
    poke.iloc[pos:, :4] *= 1.25
    f2    = session_features(poke, symbol)
    same  = (tgt in f2.index) and np.allclose(
        f2.loc[tgt, FEATURES].to_numpy(dtype=float),
        feat.loc[tgt, FEATURES].to_numpy(dtype=float), equal_nan=True)
    checks.append(("features ignore the entry bar and everything after it",
                   bool(same)))

    # 2b. and the pre-open bars of the same session ARE used - otherwise the
    #     feature set is weaker than the EMA gate it is meant to improve on
    poke2 = bars.copy()
    poke2.iloc[max(pos - 3, 0):pos, :4] *= 1.25
    f3    = session_features(poke2, symbol)
    moved = (tgt in f3.index) and not np.allclose(
        f3.loc[tgt, FEATURES].to_numpy(dtype=float),
        feat.loc[tgt, FEATURES].to_numpy(dtype=float), equal_nan=True)
    checks.append(("features DO use the pre-open bars that already closed",
                   bool(moved)))

    # 3. walk-forward must never train on anything it scores
    p, folds, imp = walk_forward_rf(feat, min_train=300, retrain_every=150)
    checks.append((f"every fold trains strictly before it predicts ({len(folds)} folds)",
                   all(tr_hi <= te_lo for _tr_lo, tr_hi, te_lo, _te_hi in folds)))

    # 4. no prediction exists before the first model is trained
    checks.append(("no prediction before min_train",
                   bool(p.iloc[:300].isna().all() and p.iloc[300:].notna().all())))

    # 5. THE LEAKAGE TEST: shuffled labels must score at the base rate
    ps, _f, _i = walk_forward_rf(feat, min_train=300, retrain_every=150, shuffle_y=True)
    m    = ps.notna()
    yy   = feat.loc[m, "_y"].astype(int)
    accs = float(((ps[m] > 0.5).astype(int) == yy).mean())
    base = float(max(yy.mean(), 1 - yy.mean()))
    checks.append((f"shuffled labels score at chance ({100*accs:.1f}% vs base "
                   f"{100*base:.1f}%)", accs <= base + 0.04))

    # 6. same seed, same answer
    p2, _f2, _i2 = walk_forward_rf(feat, min_train=300, retrain_every=150)
    checks.append(("reproducible for a fixed seed",
                   bool(np.allclose(p.to_numpy(), p2.to_numpy(), equal_nan=True))))

    # 7. the gate is subtractive, like every other gate in this notebook
    st    = rf_state(bars, symbol, feat=feat, p_up=p)
    sig   = coin_flip_signals(bars, symbol, tf="60m")
    g     = gate_signals(sig, st, symbol)
    kept  = g["position"] != 0
    checks.append(("gate never reverses a flip, and only trades on agreement",
                   bool((g.loc[kept, "position"] == g.loc[kept, "coin"]).all()
                        and (g.loc[kept, "position"] == g.loc[kept, "trend"]).all())))

    # 8. a wide margin makes the forest refuse to call anything
    checks.append(("an unreachable margin produces no trades",
                   int((gate_signals(sig, rf_state(bars, symbol, margin=0.6, feat=feat,
                                                   p_up=p), symbol)["position"] != 0).sum()) == 0))

    # 9. importances are a real distribution over the features
    checks.append(("feature importances sum to 1 over the given features",
                   abs(float(imp.sum()) - 1.0) < 1e-6 and len(imp) == len(FEATURES)))

    print(f"random forest gate self-test  ({len(feat):,} synthetic sessions, "
          f"{len(FEATURES)} features)")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [n for n, ok in checks if not ok]
    if failed:
        raise AssertionError(f"random forest self-test FAILED: {failed}")
    print("  the forest is trained walk-forward and cannot see the session it scores")
    return feat


_ = check_rf()

# ---- run it on the longest real sample --------------------------------------
_rf_src = None
for _lab in ("/NQ 1m", "/NQ 1h"):
    _s = (_sets if "_sets" in globals() else collect_assets(quiet=True)).get(_lab)
    if _s is not None:
        _rf_src = (_lab, _s[0], _s[1])
        break
if _rf_src is None:
    raise RuntimeError("no dataset - run Module.2/3 first")

print()
rf_table, rf_feat, rf_p, rf_imp, rf_st = compare_rf(_rf_src[2], _rf_src[1])
print()
print(rf_table.round(2).to_string())

_sig_now = coin_flip_signals(_rf_src[2], _rf_src[1], tf=_rf_src[2].attrs.get("tf"))
_tr_now, _, _ = run_session_trades(_rf_src[2], _sig_now, _rf_src[1])
_ = best_worst(_tr_now, rf_feat)

# Deeper cuts:
#   compare_rf(nq_1m, "/NQ", min_train=1200, retrain_every=250)
#   rf_imp.to_frame("importance")
#   walk_forward_rf(rf_feat, shuffle_y=True)     # the leakage control, by hand
